# 02 — Northeast BC Geological Carbon Storage Atlas Exploration

## Source

**Dataset:** Northeast BC Geological Carbon Capture and Storage Atlas  
**Publisher:** Geoscience BC  
**Prepared by:** Canadian Discovery Ltd.  
**Report:** Geoscience BC Report 2023-04  
**Year:** 2023

## Purpose

This notebook explores the native structure of the Northeast BC Geological
Carbon Capture and Storage Atlas prior to bronze-layer ingestion or
cross-dataset harmonization.

The objectives are to:

1. inventory the Appendix E spatial and cartographic files;
2. identify the geological and cartographic roles of the supplied shapefiles;
3. inspect the Appendix C pool and aquifer database;
4. identify relationships between Appendix C records and Appendix E geometries;
5. characterize source identifiers, attributes, CRS, geometry types, and data quality;
6. distinguish reusable geological entities from map-specific supporting layers;
7. document source-native semantics before defining a canonical storage schema.

No source data are modified in this notebook.

## Source Metadata

**Dataset:** Northeast BC Geological Carbon Capture and Storage Atlas  
**Report:** Geoscience BC Report 2023-04  
**Prepared by:** Canadian Discovery Ltd.  
**Prepared for:** Geoscience BC  
**Publication date:** January 2023  
**Study area:** Northeastern British Columbia, Canada  
**Study area extent:** >130,000 km²  
**Primary storage types assessed:**  
- Depleted and nearly depleted natural gas pools
- Deep saline aquifers

### Source Links

**Geoscience BC project page**  
https://www.geosciencebc.com/projects/2022-001/

**Main atlas report — Geoscience BC Report 2023-04**  
https://www.geosciencebc.com/i/project_data/GBCReport2023-04/NEBC%20Geological%20Carbon%20Capture%20and%20Storage%20Atlas%20GBC%20Report%202023-04.pdf

**Appendix C — Pool Storage Database & Aquifer Storage Summary**  
https://www.geosciencebc.com/i/project_data/GBCReport2023-04/Appendix%20C%20-%20Pool%20Storage%20Database%20%26%20Aquifer%20Storage%20Summary.xlsx

**Appendix E — Additional Maps and Shapefiles**  
https://www.geosciencebc.com/i/project_data/GBCReport2023-04/APPENDIX%20E%20-%20ADDITIONAL%20MAPS%20AND%20SHAPEFILES.zip

### Associated Digital Data

The atlas is accompanied by two primary digital data products used in this
notebook:

1. **Appendix C — Pool Storage Database & Aquifer Storage Summary**
   - Current CO2 Storage Candidates
   - Future CO2 Storage Candidates
   - Oil Pools for CO2-EOR Evaluation
   - Aquifer Storage Summary

2. **Appendix E — Additional Maps and Shapefiles**
   - Formation-specific map directories
   - Formation-specific shapefile directories
   - General pool shapefiles
   - Geological, reservoir, storage, and cartographic support layers

### Geological Organization

The atlas organizes potential storage reservoirs primarily by geological
formation, or by groups of related formations interpreted as a
**storage complex**. A storage complex includes the principal storage
reservoir or reservoirs together with relevant sealing units above and below.

Formation chapters included in Appendix E span:

- Peace River (Paddy/Cadotte)
- Spirit River (Notikewin/Falher)
- Bluesky
- Cadomin-Gething
- Nikanassin-Dunlevy
- Baldonnel-Pardonet
- Charlie Lake
- Halfway
- Belloy
- Debolt
- Jean Marie
- Middle Devonian Carbonates

### Storage-Potential Terminology

The atlas uses the term **storage potential** rather than commercially
established storage capacity.

At the scoping level used in this study:

- **Depleted pool storage potential** is estimated primarily from historical
  hydrocarbon production and reservoir conditions.
- **Aquifer storage potential** is estimated from mapped pore volume,
  porosity, CO2 density, and assumed storage-efficiency factors.
- Aquifer estimates are reported using P10, P50, and P90 effective
  storage-potential cases.

These values should therefore be treated as regional screening estimates
rather than directly as proven, permitted, or commercially injectable
storage capacity.

### Spatial Reference

Atlas maps indicate:

**NAD 1983 / UTM Zone 10N**

The native coordinate reference system of each supplied shapefile will be
verified directly from the source files before reprojection or spatial
standardization.

### Exploration Principle

This notebook preserves the native source structure and terminology.

No field renaming, reprojection, geometry modification, filtering,
cross-source harmonization, or conversion to a canonical storage schema is
performed until the structure and semantics of the source data are understood.

In [1]:
# ---------------------------------------------------------------------------
# Imports and source paths
# ---------------------------------------------------------------------------

from pathlib import Path
from collections import Counter

import geopandas as gpd
import pandas as pd


# ---------------------------------------------------------------------------
# Source directories
# ---------------------------------------------------------------------------

ROOT = Path(
    r"C:\Users\aviga\Research\potential data\Storage\Northeast BC_CDL"
)

APPENDIX_E = ROOT / "APPENDIX E - ADDITIONAL MAPS AND SHAPEFILES"

APPENDIX_C = (
    ROOT
    / "Appendix C - Pool Storage Database & Aquifer Storage Summary.xlsx"
)


# ---------------------------------------------------------------------------
# Basic validation
# ---------------------------------------------------------------------------

print("Root directory exists: ", ROOT.exists())
print("Appendix E exists:      ", APPENDIX_E.exists())
print("Appendix C exists:      ", APPENDIX_C.exists())

Root directory exists:  True
Appendix E exists:       True
Appendix C exists:       True


In [2]:
# ---------------------------------------------------------------------------
# Appendix E file inventory
# ---------------------------------------------------------------------------

all_files = sorted(
    path
    for path in APPENDIX_E.rglob("*")
    if path.is_file()
)

print(f"Total files found: {len(all_files):,}")


# ---------------------------------------------------------------------------
# Count files by extension
# ---------------------------------------------------------------------------

extension_counts = Counter(
    path.suffix.lower()
    for path in all_files
)

print("\nFiles by extension:")
for suffix, count in sorted(extension_counts.items()):
    label = suffix if suffix else "[no extension]"
    print(f"  {label:<10} {count:>6,}")


# ---------------------------------------------------------------------------
# Identify shapefiles
# ---------------------------------------------------------------------------

shapefiles = sorted(APPENDIX_E.rglob("*.shp"))

print(f"\nShapefiles found: {len(shapefiles):,}")

for path in shapefiles:
    print("  ", path.relative_to(APPENDIX_E))

Total files found: 933

Files by extension:
  .cpg          109
  .dbf          109
  .docx           2
  .pdf           59
  .prj          109
  .sbn          109
  .sbx          109
  .shp          109
  .shx          109
  .xml          109

Shapefiles found: 109
   CH10_BALDONNEL-PARDONET Maps and shapefiles\SHAPEFILES\GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83UTM10_PG.shp
   CH10_BALDONNEL-PARDONET Maps and shapefiles\SHAPEFILES\GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp
   CH10_BALDONNEL-PARDONET Maps and shapefiles\SHAPEFILES\GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp
   CH10_BALDONNEL-PARDONET Maps and shapefiles\SHAPEFILES\GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM10_PG.shp
   CH10_BALDONNEL-PARDONET Maps and shapefiles\SHAPEFILES\GBCS_BALDONNEL_BCOGC_CO2_STORAGE_POOL_CANDIDATES_83UTM10_PG.shp
   CH10_BALDONNEL-PARDONET Maps and shapefiles\SHAPEFILES\GBCS_BALDONNEL_BCOGC_CO2_STORAGE_POOL_NON_CANDIDATES_83UTM10_PG.shp
   CH10_BALDONNEL-PARDONET Maps and shapefil

In [3]:
# ---------------------------------------------------------------------------
# Classify Appendix E shapefiles from filenames
# ---------------------------------------------------------------------------

def classify_shapefile(filename: str) -> str:
    name = filename.upper()

    if "AQUIFER" in name:
        return "aquifer"
    if "CO2_STORAGE_POOL_CANDIDATE" in name:
        return "pool_candidates"
    if "CO2_STORAGE_POOL_NON_CANDIDATE" in name:
        return "pool_non_candidates"
    if "COMMINGLED" in name:
        return "commingled_pools"
    if "POOL_NET_RES" in name:
        return "pool_net_reservoir"
    if "NET_RES" in name:
        return "net_reservoir"
    if "STRUCTURE_ELEV" in name:
        return "structure_elevation"
    if "ABSOL_PRESSURE" in name:
        return "pressure"
    if "ISOTHERM" in name:
        return "temperature"
    if "TVD" in name:
        return "depth"
    if "TRANSITION_TO_SHALE" in name:
        return "lithologic_transition"
    if "ISOLATED_DEEP_BASIN_WET_ZONE" in name:
        return "deep_basin_wet_zone"
    if "GHG_EMISS" in name:
        return "emissions_context"
    if "POOL_CONTOURS" in name:
        return "general_pool_contours"
    if "POOLS_CO2_STORAGE" in name:
        return "general_storage_pools"

    return "other"


shp_inventory = pd.DataFrame(
    {
        "chapter": [
            path.relative_to(APPENDIX_E).parts[0]
            for path in shapefiles
        ],
        "filename": [
            path.name
            for path in shapefiles
        ],
        "relative_path": [
            str(path.relative_to(APPENDIX_E))
            for path in shapefiles
        ],
    }
)

shp_inventory["layer_class"] = (
    shp_inventory["filename"]
    .apply(classify_shapefile)
)

display(shp_inventory)

,chapter,filename,relative_path,layer_class
0,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...,aquifer
1,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...,aquifer
2,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...,aquifer
3,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...,aquifer
4,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_BCOGC_CO2_STORAGE_POOL_CANDIDAT...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...,pool_candidates
...,...,...,...,...
104,CH9_NIKANASSIN-DUNLEVY Maps and shapefiles,GBCS_NIKANASSIN_STRUCTURE_ELEV_CI_100_83UTM10_...,CH9_NIKANASSIN-DUNLEVY Maps and shapefiles\SHA...,structure_elevation
105,CH9_NIKANASSIN-DUNLEVY Maps and shapefiles,GBCS_NIKANASSIN_TVD_800_83UTM10_PL.shp,CH9_NIKANASSIN-DUNLEVY Maps and shapefiles\SHA...,depth
106,GENERAL,BCOGC_Pool_Contours_CDL.shp,GENERAL\SHAPEFILES pools\BCOGC_Pool_Contours_C...,general_pool_contours
107,GENERAL,BCOGC_POOLS_CO2_STORAGE_CDL_83UTM10_PG.shp,GENERAL\SHAPEFILES pools\BCOGC_POOLS_CO2_STORA...,general_storage_pools


In [4]:
print("Shapefiles by layer class:\n")

display(
    shp_inventory["layer_class"]
    .value_counts()
    .rename_axis("layer_class")
    .reset_index(name="count")
)

Shapefiles by layer class:



,layer_class,count
0,aquifer,25
1,pool_candidates,14
2,pool_non_candidates,14
3,pool_net_reservoir,12
4,net_reservoir,9
5,structure_elevation,9
6,pressure,7
7,depth,6
8,commingled_pools,4
9,temperature,4


In [5]:
print("Shapefiles by chapter and layer class:\n")

chapter_layer_summary = pd.crosstab(
    shp_inventory["chapter"],
    shp_inventory["layer_class"],
)

display(chapter_layer_summary)

Shapefiles by chapter and layer class:



layer_class,aquifer,commingled_pools,deep_basin_wet_zone,depth,emissions_context,general_pool_contours,general_storage_pools,lithologic_transition,net_reservoir,pool_candidates,pool_net_reservoir,pool_non_candidates,pressure,structure_elevation,temperature
chapter,,,,,,,,,,,,,,,
CH10_BALDONNEL-PARDONET Maps and shapefiles,4,1,0,0,0,0,0,0,1,1,1,1,0,1,0
CH11_CHARLIE LAKE Maps and shapefiles,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0
CH12_HALFWAY Maps and shapefiles,3,0,0,0,0,0,0,0,1,1,1,1,1,1,0
CH13_BELLOY Maps and shapefiles,3,0,0,1,0,0,0,0,1,1,1,1,1,1,0
CH14_DEBOLT Maps and shapefiles,2,0,0,1,0,0,0,0,1,1,1,1,1,1,0
CH15_JEAN MARIE Maps and shapefiles,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0
CH16_MID-DEVONIAN CARBONATES Maps and shapefiles,4,0,0,0,0,0,0,0,1,1,1,1,0,1,0
CH5_PEACE RIVER Maps and shapefiles,1,0,1,1,0,0,0,1,1,1,1,1,1,1,1
CH6_SPIRIT RIVER Maps and shapefiles,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0


In [6]:
other_layers = shp_inventory[
    shp_inventory["layer_class"] == "other"
]

print(f"Unclassified shapefiles: {len(other_layers)}")

display(other_layers)

Unclassified shapefiles: 0


,chapter,filename,relative_path,layer_class


In [7]:
# ---------------------------------------------------------------------------
# Representative shapefile inspection
# ---------------------------------------------------------------------------

representative_classes = [
    "aquifer",
    "pool_candidates",
    "pool_non_candidates",
    "pool_net_reservoir",
    "general_pool_contours",
    "general_storage_pools",
]

representative_rows = (
    shp_inventory[
        shp_inventory["layer_class"].isin(representative_classes)
    ]
    .groupby("layer_class", as_index=False)
    .first()
)

display(
    representative_rows[
        ["layer_class", "chapter", "filename", "relative_path"]
    ]
)

,layer_class,chapter,filename,relative_path
0,aquifer,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...
1,general_pool_contours,GENERAL,BCOGC_Pool_Contours_CDL.shp,GENERAL\SHAPEFILES pools\BCOGC_Pool_Contours_C...
2,general_storage_pools,GENERAL,BCOGC_POOLS_CO2_STORAGE_CDL_83UTM10_PG.shp,GENERAL\SHAPEFILES pools\BCOGC_POOLS_CO2_STORA...
3,pool_candidates,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_BCOGC_CO2_STORAGE_POOL_CANDIDAT...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...
4,pool_net_reservoir,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_BCOGC_POOL_NET_RES_CNTR_83UTM10...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...
5,pool_non_candidates,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_BCOGC_CO2_STORAGE_POOL_NON_CAND...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...


In [8]:
# ---------------------------------------------------------------------------
# Inspect representative source schemas
# ---------------------------------------------------------------------------

for _, row in representative_rows.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]

    gdf = gpd.read_file(shp_path)

    print("\n" + "=" * 100)
    print(f"Layer class:   {row['layer_class']}")
    print(f"File:          {row['filename']}")
    print(f"Features:      {len(gdf):,}")
    print(f"CRS:           {gdf.crs}")
    print(f"Geometry:      {gdf.geometry.geom_type.value_counts().to_dict()}")
    print(f"Columns ({len(gdf.columns)}):")
    print(list(gdf.columns))

    display(gdf.head(3))


Layer class:   aquifer
File:          GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83UTM10_PG.shp
Features:      1
CRS:           EPSG:26910
Geometry:      {'Polygon': 1}
Columns (8):
['TH_SCap_MT', 'Eff_SCap_h', 'Eff_SCap_2', 'Eff_SCap_5', 'CO2_Phase', 'SHAPE_Leng', 'SHAPE_Area', 'geometry']


,TH_SCap_MT,Eff_SCap_h,Eff_SCap_2,Eff_SCap_5,CO2_Phase,SHAPE_Leng,SHAPE_Area,geometry
0,0.0,3.4,13.6,36.8,Supercritical,239214.17672,2.732356e+09,"POLYGON ((683127.128 6305668.001, 684919.82 62..."



Layer class:   general_pool_contours
File:          BCOGC_Pool_Contours_CDL.shp
Features:      7,955
CRS:           EPSG:3005
Geometry:      {'LineString': 7941, 'MultiLineString': 14}
Columns (17):
['OBJECTID', 'POOL_DESIG', 'FIELD_AREA', 'FIELD_AR_1', 'FORMATION_', 'FORMATIO_1', 'POOL_SEQUE', 'FLUID_TYPE', 'MAP_SCALE', 'MAP_DRAFTE', 'DIGITIZED_', 'FCODE', 'CONTOUR_VA', 'SHAPELEN', 'LINK_CODE', 'FORM_CODE', 'geometry']


,OBJECTID,POOL_DESIG,FIELD_AREA,FIELD_AR_1,FORMATION_,FORMATIO_1,POOL_SEQUE,FLUID_TYPE,MAP_SCALE,MAP_DRAFTE,DIGITIZED_,FCODE,CONTOUR_VA,SHAPELEN,LINK_CODE,FORM_CODE,geometry
0,17688261,FIREWEED/DUNLEVY/H,3540,FIREWEED,2900,DUNLEVY,H,Gas,40000,2007-03-20,2007-03-22,None,0.0,23603.762545,3540|2900H,2900,"LINESTRING (1260192.26 1328950, 1260237.77 132..."
1,17688262,FIREWEED/DUNLEVY/H,3540,FIREWEED,2900,DUNLEVY,H,Gas,40000,2007-03-20,2007-03-22,None,10.0,7199.645826,3540|2900H,2900,"LINESTRING (1264491.91 1323386.89, 1264515.44 ..."
2,17688263,OJAY/CADOTTE/A,6480,OJAY,2200,CADOTTE,A,Gas,50000,2004-01-16,2004-10-02,None,2.0,4035.115035,6480|2200A,2200,"LINESTRING (1374464.54 1083869.39, 1374320.07 ..."



Layer class:   general_storage_pools
File:          BCOGC_POOLS_CO2_STORAGE_CDL_83UTM10_PG.shp
Features:      3,674
CRS:           EPSG:3005
Geometry:      {'Polygon': 3662, 'MultiPolygon': 12}
Columns (81):
['POOL_DESIG', 'FIELD_AREA', 'FIELD_AR_1', 'FORMATION_', 'FORMATIO_1', 'POOL_SEQUE', 'FLUID_TYPE', 'LINK_CODE', 'PL_AREA_HA', 'GBCS_PL_GP', 'FORM_CODE', 'CDL_FORM', 'POOL_UID', 'AREA_CODE', 'POOL_CODE', 'AREA_NAME', 'FORM_NAME', 'POOL_NAME', 'POOL_TYPE', 'Well_COUNT', 'CUM_CND_M3', 'CUM_G_E3M3', 'CUM_OIL_M3', 'CUM_WTR_M3', 'INJ_G_E3M3', 'INJ_WTR_M3', 'DSP_G_E3M3', 'DSP_WTR_M3', 'NET_G_E3M3', 'NET_WTR_M3', 'POOL_TVD_M', 'POOL_ELE_M', 'Z_FACTOR', 'PRES_KPA', 'TEMP_DGK', 'TEMP_DGC', 'PR_GD_KPAM', 'GTG_DEGCKM', 'OIL_FVF', 'GAS_FVF', 'WTR_FVF', 'COND_FVF', 'VD_CND_M3', 'VD_G_M3', 'VD_OIL_M3', 'VD_WTR_M3', 'VD_TOT_M3', 'CO2_D_KGM3', 'CO2_PHASE', 'TSTOR_MT', 'ESTOR_MT', 'DEP_POOL', 'RES_OIL_M3', 'RES_G_E3M3', 'REC_G_PCT', 'REC_O_PCT', 'LAST_PROD', 'POR_PCT', 'PERM_MD', 'NDEP_POOL', '5YR_

...

,POOL_DESIG,FIELD_AREA,FIELD_AR_1,FORMATION_,FORMATIO_1,POOL_SEQUE,FLUID_TYPE,LINK_CODE,PL_AREA_HA,GBCS_PL_GP,...,DISC_LONG,POOL_E_MSL,POT_COMMNG,CDL_CMN_GP,WELL_DENS,POOL_TVD,CDL_PL_TYP,PL_TYPE_ED,W_DISTBELT,geometry
0,FIREWEED/BALDONNEL/E,3540,FIREWEED,4100,BALDONNEL,E,Gas,3540|4100E,3403.791884,Baldonnel-Pardonet,...,-121.59741,-488,NaN,NaN,0.752102,1320.0,Gas,No,NaN,"POLYGON ((523566.123 6297034.879, 523560.989 6..."
1,ZAREMBA/BALDONNEL/B,8900,ZAREMBA,4100,BALDONNEL,B,Gas,8900|4100B,559.116008,Baldonnel-Pardonet,...,-121.00572,-302,NaN,NaN,0.915731,1156.0,Gas,No,NaN,"POLYGON ((557883.606 6353033.215, 557870.613 6..."
2,FORT ST JOHN/BALDONNEL/C,3600,FORT ST JOHN,4100,BALDONNEL,C,Gas,3600|4100C,1055.796560,Baldonnel-Pardonet,...,-120.87615,-506,NaN,NaN,0.242471,1206.0,Gas,No,NaN,"POLYGON ((567999.901 6243758.008, 568814.289 6..."



Layer class:   pool_net_reservoir
File:          GBCS_BALDONNEL_BCOGC_POOL_NET_RES_CNTR_83UTM10_PL.shp
Features:      669
CRS:           PROJCS["NAD83 / UTM zone 10N",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-122],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometry:      {'LineString': 666, 'MultiLineString': 3}
Columns (17):
['OBJECTID', 'POOL_DESIG', 'FIELD_AREA', 'FIELD_AR_1', 'FORMATION_', 'FORMATIO_1', 'POOL_SEQUE', 'FLUID_TYPE', 'MAP_SCALE', 'MAP_DRAFTE', 'DIGITIZED_', 'FCODE', 'CONTOUR_VA', 'SHAPELEN', 'LINK_CODE', 'FORM_CODE', 'geometry']


,OBJECTID,POOL_DESIG,FIELD_AREA,FIELD_AR_1,FORMATION_,FORMATIO_1,POOL_SEQUE,FLUID_TYPE,MAP_SCALE,MAP_DRAFTE,DIGITIZED_,FCODE,CONTOUR_VA,SHAPELEN,LINK_CODE,FORM_CODE,geometry
0,17688267,BOUNDARY LAKE NORTH/BALDONNEL/C,2020,BOUNDARY LAKE NORTH,4100,BALDONNEL,C,Gas,50000,2003-11-27,2004-09-23,None,0.0,2799.693916,2020|4100C,4100,"LINESTRING (611220.307 6279837.656, 611253.848..."
1,17688321,BUBBLES/BALDONNEL/A,2200,BUBBLES,4100,BALDONNEL,A,Gas,NaN,2004-03-30,2005-02-28,None,0.0,42221.263946,2200|4100A,4100,"LINESTRING (496749.167 6338516.281, 496701.171..."
2,17688331,BOUNDARY LAKE NORTH/BALDONNEL/A,2020,BOUNDARY LAKE NORTH,4100,BALDONNEL,A,Gas,50000,2001-11-19,2004-09-23,None,5.0,1665.684470,2020|4100A,4100,"LINESTRING (612205.924 6267206.388, 612257.386..."



Layer class:   pool_non_candidates
File:          GBCS_BALDONNEL_BCOGC_CO2_STORAGE_POOL_NON_CANDIDATES_83UTM10_PG.shp
Features:      175
CRS:           PROJCS["NAD83 / UTM zone 10N",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-122],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometry:      {'Polygon': 174, 'MultiPolygon': 1}
Columns (81):
['POOL_DESIG', 'FIELD_AREA', 'FIELD_AR_1', 'FORMATION_', 'FORMATIO_1', 'POOL_SEQUE', 'FLUID_TYPE', 'LINK_CODE', 'PL_AREA_HA', 'GBCS_PL_GP', 'FORM_CODE', 'CDL_FORM', 'POOL_UID', 'AREA_CODE', 'POOL_CODE', 'AREA_NAME', 'FORM_NAME', 'POOL_NAME', 'POOL_TYPE', 'Well_COUNT

,POOL_DESIG,FIELD_AREA,FIELD_AR_1,FORMATION_,FORMATIO_1,POOL_SEQUE,FLUID_TYPE,LINK_CODE,PL_AREA_HA,GBCS_PL_GP,...,DISC_LONG,POOL_E_MSL,POT_COMMNG,CDL_CMN_GP,WELL_DENS,POOL_TVD,CDL_PL_TYP,PL_TYPE_ED,W_DISTBELT,geometry
0,BUICK CREEK WEST/BALDONNEL/F,2800,BUICK CREEK WEST,4100,BALDONNEL,F,Gas,2800|4100F,849.213270,Baldonnel-Pardonet,...,-121.40553,-411,NaN,NaN,0.602911,1201.0,Gas,No,NaN,"POLYGON ((534203.278 6302674.611, 534195.797 6..."
1,ZAREMBA/BALDONNEL/A,8900,ZAREMBA,4100,BALDONNEL,A,Oil,8900|4100A,139.792671,Baldonnel-Pardonet,...,-121.00866,-306,NaN,NaN,1.831283,1144.0,Oil,No,NaN,"POLYGON ((559416.237 6351199.655, 559402.784 6..."
2,CECIL LAKE/BALDONNEL/A,2960,CECIL LAKE,4100,BALDONNEL,A,Gas,2960|4100A,527.950047,Baldonnel-Pardonet,...,-120.68579,-490,NaN,NaN,0.484894,1177.0,Gas,No,NaN,"POLYGON ((579371.855 6245574.653, 580185.858 6..."


In [9]:
# ---------------------------------------------------------------------------
# Validate Appendix C ↔ master pool shapefile linkage
# ---------------------------------------------------------------------------

MASTER_POOLS = (
    APPENDIX_E
    / "GENERAL"
    / "SHAPEFILES pools"
    / "BCOGC_POOLS_CO2_STORAGE_CDL_83UTM10_PG.shp"
)

master_pools = gpd.read_file(MASTER_POOLS)

print(f"Master pool features: {len(master_pools):,}")
print(f"Unique LINK_CODE values: {master_pools['LINK_CODE'].nunique():,}")
print(f"Duplicate LINK_CODE rows: {master_pools['LINK_CODE'].duplicated().sum():,}")
print(f"Missing LINK_CODE values: {master_pools['LINK_CODE'].isna().sum():,}")

Master pool features: 3,674
Unique LINK_CODE values: 3,481
Duplicate LINK_CODE rows: 193
Missing LINK_CODE values: 0


In [11]:
current_pools = pd.read_excel(
    APPENDIX_C,
    sheet_name="Current CO2 Storage Candidates",
)

future_pools = pd.read_excel(
    APPENDIX_C,
    sheet_name="Future CO2 Storage Candidates",
)

eor_pools = pd.read_excel(
    APPENDIX_C,
    sheet_name="Oil Pools for CO2-EOR Eval",
)

current_pools["source_class"] = "current_storage_candidate"
future_pools["source_class"] = "future_storage_candidate"
eor_pools["source_class"] = "oil_pool_eor_evaluation"

appendix_c_pools = pd.concat(
    [current_pools, future_pools, eor_pools],
    ignore_index=True,
)

In [12]:
LINK_FIELD_C = "Code for Link to Shapefile"

print(f"Appendix C pool rows: {len(appendix_c_pools):,}")
print(
    "Unique Appendix C link codes:",
    appendix_c_pools[LINK_FIELD_C].nunique(),
)
print(
    "Duplicate Appendix C link codes:",
    appendix_c_pools[LINK_FIELD_C].duplicated().sum(),
)
print(
    "Missing Appendix C link codes:",
    appendix_c_pools[LINK_FIELD_C].isna().sum(),
)

Appendix C pool rows: 1,238
Unique Appendix C link codes: 1238
Duplicate Appendix C link codes: 0
Missing Appendix C link codes: 0


In [13]:
shp_codes = set(
    master_pools["LINK_CODE"]
    .dropna()
    .astype(str)
)

xlsx_codes = set(
    appendix_c_pools[LINK_FIELD_C]
    .dropna()
    .astype(str)
)

print(f"Codes in both:              {len(shp_codes & xlsx_codes):,}")
print(f"Only in master SHP:         {len(shp_codes - xlsx_codes):,}")
print(f"Only in Appendix C:         {len(xlsx_codes - shp_codes):,}")

Codes in both:              1,238
Only in master SHP:         2,243
Only in Appendix C:         0


In [14]:
# ---------------------------------------------------------------------------
# Investigate duplicate LINK_CODE values in master pool polygons
# ---------------------------------------------------------------------------

duplicate_codes = (
    master_pools.loc[
        master_pools["LINK_CODE"].duplicated(keep=False),
        "LINK_CODE"
    ]
    .value_counts()
    .sort_values(ascending=False)
)

print(f"Duplicated LINK_CODE values: {len(duplicate_codes):,}")
print(f"Rows involved in duplicates: {duplicate_codes.sum():,}")

display(
    duplicate_codes
    .rename_axis("LINK_CODE")
    .reset_index(name="row_count")
    .head(25)
)

Duplicated LINK_CODE values: 193
Rows involved in duplicates: 386


,LINK_CODE,row_count
0,8400|4800B,2
1,8400|4805A,2
2,7600|2900K,2
3,5850|4800D,2
4,2960|4520B,2
5,8400|4805D,2
6,8900|4100C,2
7,0700|4805A,2
8,2800|2900B,2
9,8400|4800A,2


In [15]:
# ---------------------------------------------------------------------------
# Compare attributes within duplicated pool codes
# ---------------------------------------------------------------------------

duplicate_rows = master_pools[
    master_pools["LINK_CODE"].isin(duplicate_codes.index)
].copy()

attribute_columns = [
    col
    for col in master_pools.columns
    if col != "geometry"
]

duplicate_attribute_variation = []

for link_code, group in duplicate_rows.groupby("LINK_CODE"):
    varying_fields = [
        col
        for col in attribute_columns
        if group[col].nunique(dropna=False) > 1
    ]

    duplicate_attribute_variation.append(
        {
            "LINK_CODE": link_code,
            "row_count": len(group),
            "varying_field_count": len(varying_fields),
            "varying_fields": varying_fields,
        }
    )

duplicate_attribute_variation = pd.DataFrame(
    duplicate_attribute_variation
)

display(
    duplicate_attribute_variation
    .sort_values(
        ["varying_field_count", "row_count"],
        ascending=False,
    )
    .head(25)
)

,LINK_CODE,row_count,varying_field_count,varying_fields
8,0760|4800C,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
14,1400|7400B,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
19,2000|4800M,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
20,2000|6200L,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
23,2020|4800D,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
25,2020|4800I,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
26,2020|4800L,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
27,2020|4900A,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
29,2400|2900A,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"
30,2400|2900B,2,3,"[FLUID_TYPE, PL_AREA_HA, WELL_DENS]"


In [17]:
print(
    "Duplicate codes with identical non-geometry attributes:",
    (
        duplicate_attribute_variation["varying_field_count"] == 0
    ).sum(),
)

print(
    "Duplicate codes with differing attributes:",
    (
        duplicate_attribute_variation["varying_field_count"] > 0
    ).sum(),
)

Duplicate codes with identical non-geometry attributes: 3
Duplicate codes with differing attributes: 190


In [18]:
# ---------------------------------------------------------------------------
# Inspect duplicated master-pool identifiers
# ---------------------------------------------------------------------------

duplicate_sample_codes = duplicate_codes.index[:10]

duplicate_sample = (
    master_pools[
        master_pools["LINK_CODE"].isin(duplicate_sample_codes)
    ][
        [
            "LINK_CODE",
            "POOL_DESIG",
            "FIELD_AREA",
            "FIELD_AR_1",
            "FORMATION_",
            "FORMATIO_1",
            "POOL_SEQUE",
            "FLUID_TYPE",
            "POOL_UID",
            "AREA_CODE",
            "POOL_CODE",
            "POOL_NAME",
            "POOL_TYPE",
            "PL_AREA_HA",
            "WELL_DENS",
        ]
    ]
    .sort_values(["LINK_CODE", "FLUID_TYPE"])
)

display(duplicate_sample)

,LINK_CODE,POOL_DESIG,FIELD_AREA,FIELD_AR_1,FORMATION_,FORMATIO_1,POOL_SEQUE,FLUID_TYPE,POOL_UID,AREA_CODE,POOL_CODE,POOL_NAME,POOL_TYPE,PL_AREA_HA,WELL_DENS
101,0700|4805A,BEAVERDAM/LOWER HALFWAY/A,0700,BEAVERDAM,4805,LOWER HALFWAY,A,Gas,700-4805-A,700.0,A,Beaverdam Lower Halfway A,Oil,564.711630,0.906657
2428,0700|4805A,BEAVERDAM/LOWER HALFWAY/A,0700,BEAVERDAM,4805,LOWER HALFWAY,A,Oil,700-4805-A,700.0,A,Beaverdam Lower Halfway A,Oil,564.711630,0.906657
1965,2800|2900B,BUICK CREEK WEST/DUNLEVY/B,2800,BUICK CREEK WEST,2900,DUNLEVY,B,Gas,2800-2900-B,2800.0,B,Buick Creek West Dunlevy B,Oil,3399.390917,0.677768
110,2800|2900B,BUICK CREEK WEST/DUNLEVY/B,2800,BUICK CREEK WEST,2900,DUNLEVY,B,Oil,2800-2900-B,2800.0,B,Buick Creek West Dunlevy B,Oil,70.813578,32.536133
170,2960|4520B,CECIL LAKE/CECIL/B,2960,CECIL LAKE,4520,CECIL,B,Gas,2960-4520-B,2960.0,B,Cecil Lake Cecil B,Oil,264.221233,0.968885
36,2960|4520B,CECIL LAKE/CECIL/B,2960,CECIL LAKE,4520,CECIL,B,Oil,2960-4520-B,2960.0,B,Cecil Lake Cecil B,Oil,264.221233,0.968885
29,5850|4800D,MARTIN/HALFWAY/D,5850,MARTIN,4800,HALFWAY,D,Gas,5850-4800-D,5850.0,D,Martin Halfway D,Gas,558.755214,0.458161
3162,5850|4800D,MARTIN/HALFWAY/D,5850,MARTIN,4800,HALFWAY,D,Oil,5850-4800-D,5850.0,D,Martin Halfway D,Gas,558.761189,0.458156
2734,7600|2900K,RIGEL/DUNLEVY/K,7600,RIGEL,2900,DUNLEVY,K,Gas,7600-2900-K,7600.0,K,Rigel Dunlevy K,Gas,283.808680,0.902016
26,7600|2900K,RIGEL/DUNLEVY/K,7600,RIGEL,2900,DUNLEVY,K,Oil,7600-2900-K,7600.0,K,Rigel Dunlevy K,Gas,141.896875,1.804127


In [19]:
# ---------------------------------------------------------------------------
# Candidate unique identifiers in master pool layer
# ---------------------------------------------------------------------------

candidate_keys = [
    "POOL_UID",
    "LINK_CODE",
    "POOL_DESIG",
]

for field in candidate_keys:
    print(
        f"{field:<12}",
        f"rows={len(master_pools):,}",
        f"unique={master_pools[field].nunique(dropna=True):,}",
        f"missing={master_pools[field].isna().sum():,}",
        f"duplicates={master_pools[field].duplicated().sum():,}",
    )

POOL_UID     rows=3,674 unique=2,532 missing=958 duplicates=1,141
LINK_CODE    rows=3,674 unique=3,481 missing=0 duplicates=193
POOL_DESIG   rows=3,674 unique=3,481 missing=0 duplicates=193


In [ ]:
master_pools[
    ["LINK_CODE", "FLUID_TYPE"]
].duplicated().sum()

In [21]:
# ---------------------------------------------------------------------------
# Test compound identifiers for master pool features
# ---------------------------------------------------------------------------

compound_keys = {
    "LINK_CODE": ["LINK_CODE"],
    "LINK_CODE + FLUID_TYPE": ["LINK_CODE", "FLUID_TYPE"],
    "POOL_UID": ["POOL_UID"],
    "POOL_UID + FLUID_TYPE": ["POOL_UID", "FLUID_TYPE"],
}

for name, columns in compound_keys.items():
    valid = master_pools.dropna(subset=columns)

    duplicate_rows = valid.duplicated(
        subset=columns,
        keep=False,
    )

    duplicate_groups = (
        valid.loc[duplicate_rows, columns]
        .drop_duplicates()
    )

    print(f"{name}")
    print(f"  valid rows:              {len(valid):,}")
    print(f"  unique combinations:     {valid[columns].drop_duplicates().shape[0]:,}")
    print(f"  duplicated rows:         {duplicate_rows.sum():,}")
    print(f"  duplicated combinations: {len(duplicate_groups):,}")
    print()

LINK_CODE
  valid rows:              3,674
  unique combinations:     3,481
  duplicated rows:         386
  duplicated combinations: 193

LINK_CODE + FLUID_TYPE
  valid rows:              3,674
  unique combinations:     3,655
  duplicated rows:         38
  duplicated combinations: 19

POOL_UID
  valid rows:              2,716
  unique combinations:     2,532
  duplicated rows:         368
  duplicated combinations: 184

POOL_UID + FLUID_TYPE
  valid rows:              2,716
  unique combinations:     2,701
  duplicated rows:         30
  duplicated combinations: 15



In [22]:
# ---------------------------------------------------------------------------
# Inspect duplicates remaining after adding FLUID_TYPE
# ---------------------------------------------------------------------------

remaining_duplicates = master_pools[
    master_pools.duplicated(
        subset=["LINK_CODE", "FLUID_TYPE"],
        keep=False,
    )
].copy()

print(
    "Rows duplicated on LINK_CODE + FLUID_TYPE:",
    len(remaining_duplicates),
)

display(
    remaining_duplicates[
        [
            "LINK_CODE",
            "POOL_DESIG",
            "FLUID_TYPE",
            "POOL_UID",
            "POOL_TYPE",
            "PL_AREA_HA",
            "WELL_DENS",
        ]
    ]
    .sort_values(["LINK_CODE", "FLUID_TYPE"])
    .head(50)
)

Rows duplicated on LINK_CODE + FLUID_TYPE: 38


,LINK_CODE,POOL_DESIG,FLUID_TYPE,POOL_UID,POOL_TYPE,PL_AREA_HA,WELL_DENS
632,0350|7340A,ATTACHIE/BASAL KISKATINAW/A,Gas,350-7340-A,Gas,2642.500477,0.290634
1743,0350|7340A,ATTACHIE/BASAL KISKATINAW/A,Gas,350-7340-A,Gas,2642.516341,0.290632
932,0600|2700A,BEATTON RIVER WEST/GETHING/A,Gas,600-2700-A,Gas,280.677293,0.912079
1511,0600|2700A,BEATTON RIVER WEST/GETHING/A,Gas,600-2700-A,Gas,280.678992,0.912074
1090,2100|4700A,BRASSEY/ARTEX/A,Oil,2100-4700-A,Oil,219.471508,1.166438
2000,2100|4700A,BRASSEY/ARTEX/A,Oil,2100-4700-A,Oil,73.251165,3.494825
2680,2850|6225A,BURNT RIVER/BELCOURT/A,Gas,2850-6225-A,Gas,2358.197193,0.108558
2684,2850|6225A,BURNT RIVER/BELCOURT/A,Gas,2850-6225-A,Gas,2357.110605,0.108608
569,3320|4800B,CURRANT WEST/HALFWAY/B,Oil,3320-4800-B,Oil,283.710503,0.902328
2803,3320|4800B,CURRANT WEST/HALFWAY/B,Oil,3320-4800-B,Oil,70.779877,3.616847


In [23]:
# ---------------------------------------------------------------------------
# Compare geometries for remaining LINK_CODE + FLUID_TYPE duplicates
# ---------------------------------------------------------------------------

remaining_duplicates = master_pools[
    master_pools.duplicated(
        subset=["LINK_CODE", "FLUID_TYPE"],
        keep=False,
    )
].copy()

geometry_checks = []

for (link_code, fluid_type), group in remaining_duplicates.groupby(
    ["LINK_CODE", "FLUID_TYPE"]
):
    geometries = group.geometry.tolist()

    # All current duplicate groups contain two rows
    geom_a = geometries[0]
    geom_b = geometries[1]

    intersection_area = geom_a.intersection(geom_b).area
    union_area = geom_a.union(geom_b).area

    geometry_checks.append(
        {
            "LINK_CODE": link_code,
            "FLUID_TYPE": fluid_type,
            "rows": len(group),
            "geometry_equal": geom_a.equals(geom_b),
            "geometry_exact_equal": geom_a.equals_exact(
                geom_b,
                tolerance=0.001,
            ),
            "area_1": geom_a.area,
            "area_2": geom_b.area,
            "intersection_area": intersection_area,
            "union_area": union_area,
            "iou": (
                intersection_area / union_area
                if union_area > 0
                else None
            ),
        }
    )

geometry_checks = pd.DataFrame(geometry_checks)

display(geometry_checks)

,LINK_CODE,FLUID_TYPE,rows,geometry_equal,geometry_exact_equal,area_1,area_2,intersection_area,union_area,iou
0,0350|7340A,Gas,2,False,False,2.642500e+07,2.642516e+07,2.642466e+07,2.642551e+07,0.999968
1,0600|2700A,Gas,2,False,False,2.806773e+06,2.806790e+06,2.806698e+06,2.806865e+06,0.999940
2,2100|4700A,Oil,2,False,False,2.194715e+06,7.325117e+05,7.221573e+05,2.205069e+06,0.327499
3,2850|6225A,Gas,2,False,False,2.358197e+07,2.357111e+07,2.345441e+07,2.369867e+07,0.989693
4,3320|4800B,Oil,2,False,False,2.837105e+06,7.077988e+05,7.077959e+05,2.837108e+06,0.249478
5,3440|2700A,Oil,2,False,False,1.320357e+06,6.607931e+05,6.560718e+05,1.325078e+06,0.495119
6,4460|4570A,Gas,2,True,True,2.839797e+06,2.839797e+06,2.839797e+06,2.839797e+06,1.000000
7,4850|8400E,Gas,2,False,False,2.688608e+06,2.688118e+06,0.000000e+00,5.376726e+06,0.000000
8,5170|2700C,Gas,2,False,False,2.957612e+06,4.438621e+06,0.000000e+00,7.396233e+06,0.000000
9,5200|7400E,Gas,2,False,False,2.854193e+06,2.853251e+06,6.993125e+01,5.707374e+06,0.000012


In [24]:
print(
    "Exactly equal geometries:",
    geometry_checks["geometry_equal"].sum(),
)

print(
    "Near-identical geometries (IoU > 0.99):",
    (geometry_checks["iou"] > 0.99).sum(),
)

print(
    "Partially overlapping geometries:",
    (
        (geometry_checks["iou"] > 0)
        & (geometry_checks["iou"] <= 0.99)
    ).sum(),
)

print(
    "Spatially separate geometries:",
    (geometry_checks["iou"] == 0).sum(),
)

Exactly equal geometries: 3
Near-identical geometries (IoU > 0.99): 7
Partially overlapping geometries: 7
Spatially separate geometries: 5


### Duplicate Pool Geometry Interpretation

The master pool shapefile contains repeated logical pool identifiers.

Testing duplicate `LINK_CODE + FLUID_TYPE` combinations shows that the
duplicates are not uniform:

- 3 groups contain exactly equal geometries;
- 7 groups contain near-identical geometries;
- 7 groups contain partially overlapping geometries;
- 5 groups contain spatially separate geometries.

This indicates that repeated logical pool identifiers may represent a mixture
of duplicated records, alternate or revised polygon representations, and
multiple disconnected spatial components of the same geological pool.

Therefore, no automatic deduplication or dissolve operation is applied at this
stage.

For later bronze/database construction, logical pool identity and source spatial
feature identity should be treated as separate concepts.

In [25]:
# ---------------------------------------------------------------------------
# Compare Appendix C fields with master pool shapefile fields
# ---------------------------------------------------------------------------

print("Appendix C columns:")
for col in appendix_c_pools.columns:
    print("  ", col)

print("\nMaster pool shapefile columns:")
for col in master_pools.columns:
    print("  ", col)

Appendix C columns:
   Pool Name
   Field Code
   Pool Code
   Pool Sequence
   Pool Type
   Map Group
   Well Count
   Initial Pressure (kPa)
   Temperature (⁰C)
   Porosity (frac)
   Pool Datum TVD (m)
   Approximate Pool  Elevation (mSL)
   CO2 Phase
   Theoretical Storage Potential (Mt)
   Effective Storage Potential (Mt)
   Potentially Commingled?
   Cumulative Gas Production (e3m3)
   Cumulative Condensate Production (m3)
   Cumulative Oil Production (m3)
   Cumulative Water Production (m3)
   Cumulative Gas Injection (e3m3)
   Cumulative Water Injection (m3)
   Cumulative Gas Disposal (e3m3)
   Cumulative Water Disposal (m3)
   Gas Recovery Factor
   Inactive for 5+ years
   Discovery Well
   Discovery Well Latitude (NAD 83)
   Discovery Well Longitude (NAD 83)
   Within Disturbed Belt (Additional Evaluation Required)
   Code for Link to Shapefile
   source_class
   Oil Recovery Factor

Master pool shapefile columns:
   POOL_DESIG
   FIELD_AREA
   FIELD_AR_1
   FORMATION_
   FOR

In [26]:
# ---------------------------------------------------------------------------
# Check multiplicity of Appendix C-linked pools in master shapefile
# ---------------------------------------------------------------------------

appendix_codes = set(
    appendix_c_pools["Code for Link to Shapefile"]
    .dropna()
    .astype(str)
)

master_link_counts = (
    master_pools["LINK_CODE"]
    .astype(str)
    .value_counts()
)

appendix_link_multiplicity = (
    master_link_counts[
        master_link_counts.index.isin(appendix_codes)
    ]
    .rename_axis("LINK_CODE")
    .reset_index(name="master_feature_count")
)

print(
    "Appendix C codes represented once in master SHP:",
    (appendix_link_multiplicity["master_feature_count"] == 1).sum(),
)

print(
    "Appendix C codes represented multiple times in master SHP:",
    (appendix_link_multiplicity["master_feature_count"] > 1).sum(),
)

print(
    "Maximum master features for one Appendix C code:",
    appendix_link_multiplicity["master_feature_count"].max(),
)

display(
    appendix_link_multiplicity[
        appendix_link_multiplicity["master_feature_count"] > 1
    ]
    .sort_values("master_feature_count", ascending=False)
)

Appendix C codes represented once in master SHP: 1167
Appendix C codes represented multiple times in master SHP: 71
Maximum master features for one Appendix C code: 2


,LINK_CODE,master_feature_count
0,8400|4800B,2
1,7600|2900K,2
2,2960|4520B,2
3,2800|2900B,2
4,8400|4800A,2
...,...,...
66,2400|4805C,2
67,2985|4995A,2
68,8000|4580F,2
69,2020|4800D,2


In [27]:
# ---------------------------------------------------------------------------
# Inspect Appendix C-linked duplicate master features
# ---------------------------------------------------------------------------

appendix_duplicate_codes = set(
    appendix_link_multiplicity.loc[
        appendix_link_multiplicity["master_feature_count"] > 1,
        "LINK_CODE",
    ]
)

appendix_duplicate_features = (
    master_pools[
        master_pools["LINK_CODE"].isin(appendix_duplicate_codes)
    ]
    .copy()
    .sort_values(["LINK_CODE", "FLUID_TYPE"])
)

print(
    "Appendix C-linked duplicate codes:",
    len(appendix_duplicate_codes),
)

print(
    "Master rows involved:",
    len(appendix_duplicate_features),
)

display(
    appendix_duplicate_features[
        [
            "LINK_CODE",
            "POOL_DESIG",
            "FLUID_TYPE",
            "POOL_UID",
            "POOL_TYPE",
            "PL_AREA_HA",
            "Well_COUNT",
            "PRES_KPA",
            "TEMP_DGC",
            "POR_PCT",
            "POOL_TVD_M",
            "TSTOR_MT",
            "ESTOR_MT",
            "REC_G_PCT",
            "REC_O_PCT",
            "CDL_CAND",
            "PL_TYPE_ED",
        ]
    ]
    .head(100)
)

Appendix C-linked duplicate codes: 71
Master rows involved: 142


,LINK_CODE,POOL_DESIG,FLUID_TYPE,POOL_UID,POOL_TYPE,PL_AREA_HA,Well_COUNT,PRES_KPA,TEMP_DGC,POR_PCT,POOL_TVD_M,TSTOR_MT,ESTOR_MT,REC_G_PCT,REC_O_PCT,CDL_CAND,PL_TYPE_ED
1731,0200|2700A,AITKEN CREEK/GETHING/A,Gas,200-2700-A,Oil,4799.670411,19.0,10760.0,59.9,0.119,1337.886667,3.73,1.80905,-1.000000,0.978500,Y,Yes
1732,0200|2700A,AITKEN CREEK/GETHING/A,Oil,200-2700-A,Oil,4799.670411,19.0,10760.0,59.9,0.119,1337.886667,3.73,1.80905,-1.000000,0.978500,Y,Yes
2424,0400|4800B,BEATTON RIVER/HALFWAY/B,Gas,400-4800-B,Oil,911.155511,5.0,8113.0,53.9,0.165,1127.560000,0.34,0.16490,-1.000000,0.808796,Y,Yes
2125,0400|4800B,BEATTON RIVER/HALFWAY/B,Oil,400-4800-B,Oil,911.155511,5.0,8113.0,53.9,0.165,1127.560000,0.34,0.16490,-1.000000,0.808796,Y,Yes
932,0600|2700A,BEATTON RIVER WEST/GETHING/A,Gas,600-2700-A,Gas,280.677293,1.0,8373.0,43.9,0.171,1013.000000,0.05,0.03150,1.000000,-1.000000,Y,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2418,6500|4800G,OSPREY/HALFWAY/G,Oil,6500-4800-G,Oil,1695.551664,4.0,9490.0,55.9,0.196,1187.000000,0.64,0.16000,-1.000000,0.067362,Y,Yes
2876,6800|2700A,PEEJAY/GETHING/A,Gas,6800-2700-A,Gas,282.331753,1.0,7396.0,48.9,0.165,987.200000,0.51,0.44523,0.926931,-1.000000,Y,No
2704,6800|2700A,PEEJAY/GETHING/A,Oil,6800-2700-A,Gas,282.331753,1.0,7396.0,48.9,0.165,987.200000,0.51,0.44523,0.926931,-1.000000,Y,No
355,6800|4800N,PEEJAY/HALFWAY/N,Gas,6800-4800-N,Oil,282.153427,2.0,8652.0,60.9,0.102,1126.250000,0.17,0.08245,-1.000000,0.962489,Y,Yes


In [28]:
# ---------------------------------------------------------------------------
# Determine whether duplicate pairs differ by fluid type
# ---------------------------------------------------------------------------

duplicate_pair_summary = []

for link_code, group in appendix_duplicate_features.groupby("LINK_CODE"):

    duplicate_pair_summary.append(
        {
            "LINK_CODE": link_code,
            "rows": len(group),
            "fluid_types": sorted(
                group["FLUID_TYPE"]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            ),
            "unique_fluid_types": group["FLUID_TYPE"].nunique(dropna=False),
            "unique_pool_types": group["POOL_TYPE"].nunique(dropna=False),
            "unique_storage_values": group["ESTOR_MT"].nunique(dropna=False),
        }
    )

duplicate_pair_summary = pd.DataFrame(duplicate_pair_summary)

display(
    duplicate_pair_summary[
        "unique_fluid_types unique_pool_types unique_storage_values".split()
    ].value_counts()
)

unique_fluid_types  unique_pool_types  unique_storage_values
2                   1                  1                        68
1                   1                  1                         3
Name: count, dtype: int64

In [29]:
# ---------------------------------------------------------------------------
# Test whether Appendix C Pool Type resolves duplicate LINK_CODE matches
# ---------------------------------------------------------------------------

appendix_pair_keys = (
    appendix_c_pools[
        [
            "Code for Link to Shapefile",
            "Pool Type",
        ]
    ]
    .rename(
        columns={
            "Code for Link to Shapefile": "LINK_CODE",
            "Pool Type": "FLUID_TYPE",
        }
    )
)

pair_match = appendix_pair_keys.merge(
    master_pools[
        [
            "LINK_CODE",
            "FLUID_TYPE",
        ]
    ],
    on=["LINK_CODE", "FLUID_TYPE"],
    how="left",
    indicator=True,
)

print(pair_match["_merge"].value_counts())

print(
    "Duplicate LINK_CODE + FLUID_TYPE matches:",
    pair_match.duplicated(
        subset=["LINK_CODE", "FLUID_TYPE"],
        keep=False,
    ).sum(),
)

_merge
both          1234
left_only        7
right_only       0
Name: count, dtype: int64
Duplicate LINK_CODE + FLUID_TYPE matches: 6


In [30]:
# ---------------------------------------------------------------------------
# Inspect unresolved Appendix C ↔ master pool linkage cases
# ---------------------------------------------------------------------------

# Appendix C records that did not match on LINK_CODE + Pool Type
left_only = pair_match[
    pair_match["_merge"] == "left_only"
].copy()

print(f"Appendix C records unmatched on LINK_CODE + Pool Type: {len(left_only)}")

display(left_only)

Appendix C records unmatched on LINK_CODE + Pool Type: 7


,LINK_CODE,FLUID_TYPE,_merge
216,6440|4580B,Gas,left_only
361,8100|6200E,Gas,left_only
515,7600|2900B,Gas,left_only
544,8300|4805A,Gas,left_only
589,8400|4805G,Gas,left_only
930,6800|2700S,Gas,left_only
931,8100|4520B,Gas,left_only


In [33]:
# ---------------------------------------------------------------------------
# Inspect master features duplicated on LINK_CODE + FLUID_TYPE
# ---------------------------------------------------------------------------

duplicate_compound_keys = (
    master_pools[
        master_pools.duplicated(
            subset=["LINK_CODE", "FLUID_TYPE"],
            keep=False,
        )
    ]
    .sort_values(["LINK_CODE", "FLUID_TYPE"])
)

print(
    "Master rows involved in duplicate LINK_CODE + FLUID_TYPE keys:",
    len(duplicate_compound_keys),
)

display(
    duplicate_compound_keys[
        [
            "LINK_CODE",
            "POOL_DESIG",
            "FLUID_TYPE",
            "POOL_UID",
            "POOL_TYPE",
            "PL_AREA_HA",
            "Well_COUNT",
            "TSTOR_MT",
            "ESTOR_MT",
            "CDL_CAND",
            "PL_TYPE_ED",
        ]
    ]
)

Master rows involved in duplicate LINK_CODE + FLUID_TYPE keys: 38


,LINK_CODE,POOL_DESIG,FLUID_TYPE,POOL_UID,POOL_TYPE,PL_AREA_HA,Well_COUNT,TSTOR_MT,ESTOR_MT,CDL_CAND,PL_TYPE_ED
632,0350|7340A,ATTACHIE/BASAL KISKATINAW/A,Gas,350-7340-A,Gas,2642.500477,3.0,2.80,2.44440,Y,No
1743,0350|7340A,ATTACHIE/BASAL KISKATINAW/A,Gas,350-7340-A,Gas,2642.516341,3.0,2.80,2.44440,Y,No
932,0600|2700A,BEATTON RIVER WEST/GETHING/A,Gas,600-2700-A,Gas,280.677293,1.0,0.05,0.03150,Y,No
1511,0600|2700A,BEATTON RIVER WEST/GETHING/A,Gas,600-2700-A,Gas,280.678992,1.0,0.05,0.03150,Y,No
1090,2100|4700A,BRASSEY/ARTEX/A,Oil,2100-4700-A,Oil,219.471508,1.0,0.05,0.02425,F,No
2000,2100|4700A,BRASSEY/ARTEX/A,Oil,2100-4700-A,Oil,73.251165,1.0,0.05,0.02425,F,No
2680,2850|6225A,BURNT RIVER/BELCOURT/A,Gas,2850-6225-A,Gas,2358.197193,1.0,4.03,3.51819,N,No
2684,2850|6225A,BURNT RIVER/BELCOURT/A,Gas,2850-6225-A,Gas,2357.110605,1.0,4.03,3.51819,N,No
569,3320|4800B,CURRANT WEST/HALFWAY/B,Oil,3320-4800-B,Oil,283.710503,1.0,0.00,0.00000,F,No
2803,3320|4800B,CURRANT WEST/HALFWAY/B,Oil,3320-4800-B,Oil,70.779877,1.0,0.00,0.00000,F,No


In [32]:
# ---------------------------------------------------------------------------
# Check whether unmatched compound keys exist by LINK_CODE alone
# ---------------------------------------------------------------------------

left_only_codes = set(left_only["LINK_CODE"])

left_only_master = (
    master_pools[
        master_pools["LINK_CODE"].isin(left_only_codes)
    ][
        [
            "LINK_CODE",
            "POOL_DESIG",
            "FLUID_TYPE",
            "POOL_TYPE",
            "TSTOR_MT",
            "ESTOR_MT",
        ]
    ]
    .sort_values(["LINK_CODE", "FLUID_TYPE"])
)

display(left_only_master)

,LINK_CODE,POOL_DESIG,FLUID_TYPE,POOL_TYPE,TSTOR_MT,ESTOR_MT
629,6440|4580B,NORTH PINE/NORTH PINE/B,Oil,Oil,1.17,0.56745
1949,6800|2700S,PEEJAY/GETHING/S,Oil,Oil,0.04,0.01000
878,7600|2900B,RIGEL/DUNLEVY/B,Oil,Oil,0.20,0.09700
717,8100|4520B,STODDART WEST/CECIL/B,Oil,Oil,0.04,0.01000
397,8100|6200E,STODDART WEST/BELLOY/E,Oil,Oil,0.91,0.22750
2721,8300|4805A,WEASEL/LOWER HALFWAY/A,Oil,Oil,0.35,0.08750
86,8400|4805G,WILDMINT/LOWER HALFWAY/G,Oil,Oil,0.16,0.07760


### Appendix C ↔ Master Pool Linkage Interpretation

All 1,238 Appendix C pool records have a matching `LINK_CODE` in the
master pool shapefile.

`LINK_CODE` therefore functions as the primary cross-source linkage key.

Most master-pool `LINK_CODE` duplicates can be distinguished using
`FLUID_TYPE`, but this is not universally reliable:

- 71 Appendix C-linked `LINK_CODE`s correspond to two master features;
- most of these pairs differ by fluid type;
- 19 `LINK_CODE + FLUID_TYPE` combinations remain duplicated;
- 7 Appendix C records have a pool-type classification that does not match
  the master shapefile `FLUID_TYPE`.

The remaining duplicate features generally preserve the same storage and
reservoir attributes while differing primarily in geometry-derived fields
such as polygon area.

Therefore:

- `LINK_CODE` should be preserved as the authoritative logical pool linkage key;
- spatial feature identity should remain separate from logical pool identity;
- `FLUID_TYPE` may assist in resolving ambiguous geometries but should not be
  treated as part of a universal primary key;
- no deduplication or dissolve is performed during source exploration.

In [34]:
# ---------------------------------------------------------------------------
# Create one-row-per-pool comparison view
# ---------------------------------------------------------------------------

master_pool_attributes = (
    master_pools
    .drop(columns="geometry")
    .sort_values("LINK_CODE")
    .drop_duplicates(
        subset="LINK_CODE",
        keep="first",
    )
)

print(f"Master logical pools: {len(master_pool_attributes):,}")

Master logical pools: 3,481


In [35]:
# ---------------------------------------------------------------------------
# Join Appendix C to master pool attributes
# ---------------------------------------------------------------------------

pool_comparison = appendix_c_pools.merge(
    master_pool_attributes,
    left_on="Code for Link to Shapefile",
    right_on="LINK_CODE",
    how="left",
    validate="one_to_one",
)

print(f"Comparison rows: {len(pool_comparison):,}")
print(
    "Missing master matches:",
    pool_comparison["LINK_CODE"].isna().sum(),
)

Comparison rows: 1,238
Missing master matches: 0


In [36]:
# ---------------------------------------------------------------------------
# Compare selected numeric fields
# ---------------------------------------------------------------------------

numeric_field_mapping = {
    "Well Count": "Well_COUNT",
    "Initial Pressure (kPa)": "PRES_KPA",
    "Temperature (⁰C)": "TEMP_DGC",
    "Pool Datum TVD (m)": "POOL_TVD_M",
    "Theoretical Storage Potential (Mt)": "TSTOR_MT",
    "Effective Storage Potential (Mt)": "ESTOR_MT",
    "Cumulative Gas Production (e3m3)": "CUM_G_E3M3",
    "Cumulative Condensate Production (m3)": "CUM_CND_M3",
    "Cumulative Oil Production (m3)": "CUM_OIL_M3",
    "Cumulative Water Production (m3)": "CUM_WTR_M3",
    "Cumulative Gas Injection (e3m3)": "INJ_G_E3M3",
    "Cumulative Water Injection (m3)": "INJ_WTR_M3",
    "Cumulative Gas Disposal (e3m3)": "DSP_G_E3M3",
    "Cumulative Water Disposal (m3)": "DSP_WTR_M3",
    "Discovery Well Latitude (NAD 83)": "DISC_LAT",
    "Discovery Well Longitude (NAD 83)": "DISC_LONG",
}

numeric_results = []

for xlsx_field, shp_field in numeric_field_mapping.items():

    left = pd.to_numeric(
        pool_comparison[xlsx_field],
        errors="coerce",
    )

    right = pd.to_numeric(
        pool_comparison[shp_field],
        errors="coerce",
    )

    both_missing = left.isna() & right.isna()

    close = (
        (left - right).abs() < 1e-6
    ) | both_missing

    numeric_results.append(
        {
            "appendix_c_field": xlsx_field,
            "shapefile_field": shp_field,
            "matches": int(close.sum()),
            "rows": len(close),
            "match_pct": 100 * close.mean(),
        }
    )

numeric_comparison_summary = pd.DataFrame(numeric_results)

display(
    numeric_comparison_summary.sort_values(
        "match_pct",
        ascending=False,
    )
)

,appendix_c_field,shapefile_field,matches,rows,match_pct
0,Well Count,Well_COUNT,1238,1238,100.0
1,Initial Pressure (kPa),PRES_KPA,1238,1238,100.0
2,Temperature (⁰C),TEMP_DGC,1238,1238,100.0
3,Pool Datum TVD (m),POOL_TVD_M,1238,1238,100.0
4,Theoretical Storage Potential (Mt),TSTOR_MT,1238,1238,100.0
5,Effective Storage Potential (Mt),ESTOR_MT,1238,1238,100.0
6,Cumulative Gas Production (e3m3),CUM_G_E3M3,1238,1238,100.0
7,Cumulative Condensate Production (m3),CUM_CND_M3,1238,1238,100.0
8,Cumulative Oil Production (m3),CUM_OIL_M3,1238,1238,100.0
9,Cumulative Water Production (m3),CUM_WTR_M3,1238,1238,100.0


### Appendix C ↔ Master Pool Attribute Consistency

All tested numeric fields in Appendix C match the corresponding fields in the
master pool shapefile for all 1,238 linked pool records.

The comparison included:

- well count;
- initial pressure;
- temperature;
- pool datum TVD;
- theoretical storage potential;
- effective storage potential;
- cumulative gas, condensate, oil, and water production;
- cumulative gas and water injection;
- cumulative gas and water disposal; and
- discovery-well latitude and longitude.

Each field matched across 100% of the linked records.

This indicates that the Appendix C pool tables are a curated, human-readable
tabular representation of data already contained in the master pool shapefile,
rather than an independent numerical source.

For future ingestion, the master pool shapefile may therefore serve as the
primary geospatial source for pool records, while Appendix C remains useful for:

- source validation;
- understanding CDL's intended storage-candidate classifications;
- confirming field semantics and human-readable units; and
- providing the separate aquifer storage summary.

In [37]:
# ---------------------------------------------------------------------------
# Compare transformed / semantic fields
# ---------------------------------------------------------------------------

# Porosity: Appendix C stores fraction; SHP field name suggests percent
porosity_xlsx = pd.to_numeric(
    pool_comparison["Porosity (frac)"],
    errors="coerce",
)

porosity_shp = pd.to_numeric(
    pool_comparison["POR_PCT"],
    errors="coerce",
)

porosity_match = (
    (porosity_xlsx * 100 - porosity_shp).abs() < 1e-6
) | (
    porosity_xlsx.isna() & porosity_shp.isna()
)

print(
    "Porosity ×100 matches POR_PCT:",
    f"{porosity_match.sum():,} / {len(porosity_match):,}",
    f"({100 * porosity_match.mean():.2f}%)",
)

Porosity ×100 matches POR_PCT: 20 / 1,238 (1.62%)


In [38]:
# ---------------------------------------------------------------------------
# Recovery-factor scaling
# ---------------------------------------------------------------------------

for xlsx_field, shp_field in [
    ("Gas Recovery Factor", "REC_G_PCT"),
    ("Oil Recovery Factor", "REC_O_PCT"),
]:
    left = pd.to_numeric(
        pool_comparison[xlsx_field],
        errors="coerce",
    )

    right = pd.to_numeric(
        pool_comparison[shp_field],
        errors="coerce",
    )

    direct_match = (
        (left - right).abs() < 1e-6
    ) | (
        left.isna() & right.isna()
    )

    scaled_match = (
        (left * 100 - right).abs() < 1e-6
    ) | (
        left.isna() & right.isna()
    )

    print(f"\n{xlsx_field} ↔ {shp_field}")
    print(
        f"  Direct match:  {direct_match.sum():,} "
        f"({100 * direct_match.mean():.2f}%)"
    )
    print(
        f"  ×100 match:    {scaled_match.sum():,} "
        f"({100 * scaled_match.mean():.2f}%)"
    )


Gas Recovery Factor ↔ REC_G_PCT
  Direct match:  1,186 (95.80%)
  ×100 match:    0 (0.00%)

Oil Recovery Factor ↔ REC_O_PCT
  Direct match:  52 (4.20%)
  ×100 match:    0 (0.00%)


In [39]:
# ---------------------------------------------------------------------------
# Compare categorical fields
# ---------------------------------------------------------------------------

categorical_mapping = {
    "Pool Sequence": "POOL_SEQUE",
    "Map Group": "GBCS_PL_GP",
    "CO2 Phase": "CO2_PHASE",
    "Potentially Commingled?": "POT_COMMNG",
    "Inactive for 5+ years": "5YR_INACT",
    "Discovery Well": "DISC_WELL",
    "Within Disturbed Belt (Additional Evaluation Required)": "W_DISTBELT",
}

categorical_results = []

for xlsx_field, shp_field in categorical_mapping.items():

    left = (
        pool_comparison[xlsx_field]
        .fillna("__NA__")
        .astype(str)
        .str.strip()
    )

    right = (
        pool_comparison[shp_field]
        .fillna("__NA__")
        .astype(str)
        .str.strip()
    )

    match = left == right

    categorical_results.append(
        {
            "appendix_c_field": xlsx_field,
            "shapefile_field": shp_field,
            "matches": int(match.sum()),
            "rows": len(match),
            "match_pct": 100 * match.mean(),
        }
    )

categorical_comparison_summary = pd.DataFrame(
    categorical_results
)

display(categorical_comparison_summary)

,appendix_c_field,shapefile_field,matches,rows,match_pct
0,Pool Sequence,POOL_SEQUE,1238,1238,100.000000
1,Map Group,GBCS_PL_GP,1233,1238,99.596123
2,CO2 Phase,CO2_PHASE,1238,1238,100.000000
3,Potentially Commingled?,POT_COMMNG,1238,1238,100.000000
4,Inactive for 5+ years,5YR_INACT,1238,1238,100.000000
5,Discovery Well,DISC_WELL,1238,1238,100.000000
6,Within Disturbed Belt (Additional Evaluation R...,W_DISTBELT,1238,1238,100.000000


In [40]:
# ---------------------------------------------------------------------------
# Diagnose remaining Appendix C ↔ master SHP field differences
# ---------------------------------------------------------------------------

# 1. Porosity: test direct correspondence
xlsx_porosity = pd.to_numeric(
    pool_comparison["Porosity (frac)"],
    errors="coerce",
)

shp_porosity = pd.to_numeric(
    pool_comparison["POR_PCT"],
    errors="coerce",
)

porosity_direct = (
    (xlsx_porosity - shp_porosity).abs() < 1e-6
) | (
    xlsx_porosity.isna() & shp_porosity.isna()
)

print(
    "Direct porosity match:",
    f"{porosity_direct.sum():,} / {len(porosity_direct):,}",
    f"({100 * porosity_direct.mean():.2f}%)",
)


# 2. Recovery factors by Appendix C source class
for source_class, group in pool_comparison.groupby("source_class"):

    print("\n" + "=" * 80)
    print(source_class)

    for xlsx_field, shp_field in [
        ("Gas Recovery Factor", "REC_G_PCT"),
        ("Oil Recovery Factor", "REC_O_PCT"),
    ]:
        left = pd.to_numeric(group[xlsx_field], errors="coerce")
        right = pd.to_numeric(group[shp_field], errors="coerce")

        match = (
            (left - right).abs() < 1e-6
        ) | (
            left.isna() & right.isna()
        )

        print(
            f"{xlsx_field:<24}",
            f"{match.sum():,}/{len(match):,}",
            f"({100 * match.mean():.2f}%)",
        )


# 3. Inspect Map Group mismatches
map_group_left = (
    pool_comparison["Map Group"]
    .fillna("__NA__")
    .astype(str)
    .str.strip()
)

map_group_right = (
    pool_comparison["GBCS_PL_GP"]
    .fillna("__NA__")
    .astype(str)
    .str.strip()
)

map_group_mismatch = pool_comparison[
    map_group_left != map_group_right
][
    [
        "Code for Link to Shapefile",
        "Pool Name",
        "source_class",
        "Map Group",
        "GBCS_PL_GP",
    ]
]

print(
    "\nMap Group mismatches:",
    len(map_group_mismatch),
)

display(map_group_mismatch)

Direct porosity match: 1,238 / 1,238 (100.00%)

current_storage_candidate
Gas Recovery Factor      929/929 (100.00%)
Oil Recovery Factor      0/929 (0.00%)

future_storage_candidate
Gas Recovery Factor      257/257 (100.00%)
Oil Recovery Factor      0/257 (0.00%)

oil_pool_eor_evaluation
Gas Recovery Factor      0/52 (0.00%)
Oil Recovery Factor      52/52 (100.00%)

Map Group mismatches: 5


,Code for Link to Shapefile,Pool Name,source_class,Map Group,GBCS_PL_GP
117,2985|4995A,Chinchaga River Lower Charlie Lake/Montney A,current_storage_candidate,Charlie Lake,Montney
931,7660|4990A,Ring Bluesky Gething-Montney-A,future_storage_candidate,Bluesky,Montney
961,4485|4990A,Gutah Bluesky Gething-Montney-A,future_storage_candidate,Bluesky,Montney
1019,4485|4990B,Gutah Bluesky Gething-Montney-B,future_storage_candidate,Bluesky,Montney
1089,4485|4990D,Gutah Bluesky Gething-Montney-D,future_storage_candidate,Bluesky,Montney


### Pool Attribute Comparison — Conclusions

The Appendix C pool tables were compared against the master pool shapefile
using `LINK_CODE`.

All 1,238 Appendix C pool records have matching master geometries.

#### Numeric Attributes

All directly comparable numeric fields tested were identical across the two
sources, including:

- well count;
- initial pressure;
- temperature;
- pool datum TVD;
- theoretical storage potential;
- effective storage potential;
- cumulative production;
- injection and disposal volumes; and
- discovery-well coordinates.

Porosity also matched exactly between Appendix C `Porosity (frac)` and
the shapefile field `POR_PCT`.

Despite its name, `POR_PCT` therefore stores porosity as a fractional value,
not as a value from 0–100.

#### Recovery Factors

Appendix C exposes the recovery factor appropriate to the storage-screening
class:

- current storage candidates: gas recovery factor;
- future storage candidates: gas recovery factor;
- oil pools for CO2-EOR evaluation: oil recovery factor.

The master shapefile retains both gas and oil recovery-factor fields.

#### Map Group

`Map Group` and `GBCS_PL_GP` agree for 1,233 of 1,238 records.

The five differences involve mixed or composite geological units where the
master shapefile assigns the pool to `Montney`, while Appendix C assigns the
pool to the atlas formation grouping used for storage evaluation
(e.g. Bluesky or Charlie Lake).

These fields should therefore be preserved separately until formation and
stratigraphic normalization rules are defined.

#### Interpretation

Appendix C is primarily a curated, human-readable storage-oriented view of
attributes already present in the richer master pool shapefile.

For pool ingestion, the master shapefile appears to be the more complete
native geospatial source, while Appendix C remains useful for:

- validating field semantics;
- identifying current, future, and EOR screening classes;
- preserving atlas-specific map-group assignments; and
- providing human-readable documentation of the source fields.

In [41]:
# ---------------------------------------------------------------------------
# Aquifer source inventory
# ---------------------------------------------------------------------------

aquifer_shapefiles = shp_inventory[
    shp_inventory["layer_class"] == "aquifer"
].copy()

print(f"Aquifer shapefiles: {len(aquifer_shapefiles):,}")

display(
    aquifer_shapefiles[
        ["chapter", "filename", "relative_path"]
    ]
)

Aquifer shapefiles: 25


,chapter,filename,relative_path
0,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...
1,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...
2,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...
3,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM...,CH10_BALDONNEL-PARDONET Maps and shapefiles\SH...
15,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_P...,CH12_HALFWAY Maps and shapefiles\SHAPEFILES\GB...
16,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp,CH12_HALFWAY Maps and shapefiles\SHAPEFILES\GB...
17,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp,CH12_HALFWAY Maps and shapefiles\SHAPEFILES\GB...
24,CH13_BELLOY Maps and shapefiles,GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp,CH13_BELLOY Maps and shapefiles\SHAPEFILES\GBC...
25,CH13_BELLOY Maps and shapefiles,GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp,CH13_BELLOY Maps and shapefiles\SHAPEFILES\GBC...
26,CH13_BELLOY Maps and shapefiles,GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp,CH13_BELLOY Maps and shapefiles\SHAPEFILES\GBC...


In [42]:
aquifers = pd.read_excel(
    APPENDIX_C,
    sheet_name="Aquifer Storage Summary",
)

print(f"Appendix C aquifer rows: {len(aquifers):,}")
display(aquifers)

Appendix C aquifer rows: 25


,Formation,Aquifer Name,Type,Thickness Range (m),Pressure Range (MPa),Temperature Range (C),Porosity Range (%),CO2 Phase,P10 Effective Storage Potential at 0.5% (Mt),P50 Effective Storage Potential at 2% (Mt),P90 Effective Storage Potential at 5.4% (Mt)
0,Peace River,Dawson Creek,Aquifer,5-40,3.9-7.5,27-48,16-28%,Gas,3.040814,12.163257,32.840795
1,Bluesky,Chinchaga-Dahl,Aquifer,2-10,5.7-7.9,43-60,9-21%,Mostly Gas,1.524081,6.096325,16.460077
2,Bluesky,Doe-Airport,Aquifer,2-35,6.6-11.1,27-56,8-22%,Mostly Supercritical,9.051615,36.206459,97.757439
3,Cadomin-Gething,Parkland-Muskrat,Aquifer,10-50,8.0-11.1,29-54,6-17%,Supercritical,26.207647,104.830588,283.042587
4,Cadomin-Gething,Stoddart West,Aquifer,10-37,6.4-13.0,22-54,7-13%,Mostly Supercritical,16.244408,64.977632,175.439606
5,Cadomin-Gething,Sunrise-Doe,Aquifer,10-40,10.8-15.1,41-67,7-14%,Supercritical,10.677841,42.711364,115.320683
6,Nikanassin-Dunlevy,Beg-Siphon,Aquifer,10-30,8.4-13.7,45-66,7-12%,Supercritical,5.934243,23.736970,64.089819
7,Nikanassin-Dunlevy,Blueberry-Two Rivers,Aquifer,10-110,7.0-15.0,26-60,7-14%,Mostly Supercritical,110.150721,440.602886,1189.627792
8,Nikanassin-Dunlevy,Brassey-Cutbank,Aquifer,10-60,11.5-25.0,48-100,7-14%,Supercritical,23.928082,95.712327,258.423282
9,Baldonnel-Pardonet,Boundary Lake-Osprey,Aquifer,5-12,8.1-10.0,46-60,8-15%,Supercritical,3.405266,13.621065,36.776876


In [ ]:
# ---------------------------------------------------------------------------
# Inspect all aquifer shapefiles
# ---------------------------------------------------------------------------

aquifer_records = []

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]
    gdf = gpd.read_file(shp_path)

    aquifer_records.append(
        {
            "chapter": row["chapter"],
            "filename": row["filename"],
            "feature_count": len(gdf),
            "crs": gdf.crs.to_string() if gdf.crs else None,
            "geometry_types": sorted(
                gdf.geometry.geom_type.dropna().unique().tolist()
            ),
            "columns": list(gdf.columns),
            "column_count": len(gdf.columns),
        }
    )

aquifer_structure = pd.DataFrame(aquifer_records)

display(
    aquifer_structure[
        [
            "chapter",
            "filename",
            "feature_count",
            "crs",
            "geometry_types",
            "column_count",
        ]
    ]
)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,chapter,filename,feature_count,crs,geometry_types,column_count
0,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,1,EPSG:26910,[Polygon],8
1,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp,1,EPSG:26910,[Polygon],8
2,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,1,EPSG:26910,[MultiPolygon],8
3,CH10_BALDONNEL-PARDONET Maps and shapefiles,GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM...,1,EPSG:26910,[Polygon],8
4,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_P...,1,EPSG:26910,[MultiPolygon],9
5,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp,1,EPSG:26910,[MultiPolygon],9
6,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp,1,EPSG:26910,[Polygon],9
7,CH13_BELLOY Maps and shapefiles,GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp,1,EPSG:26910,[Polygon],8
8,CH13_BELLOY Maps and shapefiles,GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp,1,EPSG:26910,[Polygon],8
9,CH13_BELLOY Maps and shapefiles,GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp,1,EPSG:26910,[Polygon],8


In [44]:
# ---------------------------------------------------------------------------
# Compare aquifer shapefile schemas
# ---------------------------------------------------------------------------

schema_counts = (
    aquifer_structure["columns"]
    .apply(tuple)
    .value_counts()
)

print("Distinct aquifer schemas:", len(schema_counts))

display(
    schema_counts
    .rename_axis("schema")
    .reset_index(name="file_count")
)

Distinct aquifer schemas: 7


,schema,file_count
0,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, Eff_Scap_...",10
1,"(TH_SCap_MT, Eff_SCap_h, Eff_SCap_2, Eff_SCap_...",7
2,"(NAME, Th_SCap_MT, Eff_SCap_h, Eff_SCap_2, Eff...",3
3,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, Eff_Scap_...",2
4,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, Eff_Scap_...",1
5,"(REC_ID, Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, E...",1
6,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_5, Eff_Scap_...",1


In [45]:
print(
    "Files with exactly one feature:",
    (aquifer_structure["feature_count"] == 1).sum(),
)

print(
    "Files with multiple features:",
    (aquifer_structure["feature_count"] > 1).sum(),
)

print(
    "Maximum features in one aquifer shapefile:",
    aquifer_structure["feature_count"].max(),
)

Files with exactly one feature: 23
Files with multiple features: 2
Maximum features in one aquifer shapefile: 3


In [46]:
# ---------------------------------------------------------------------------
# Aquifer structure exceptions
# ---------------------------------------------------------------------------

print("CRS distribution:")
display(
    aquifer_structure["crs"]
    .value_counts(dropna=False)
    .rename_axis("crs")
    .reset_index(name="file_count")
)

print("\nColumn-count distribution:")
display(
    aquifer_structure["column_count"]
    .value_counts()
    .sort_index()
    .rename_axis("column_count")
    .reset_index(name="file_count")
)

print("\nAquifers with multiple features:")
display(
    aquifer_structure[
        aquifer_structure["feature_count"] > 1
    ][
        [
            "chapter",
            "filename",
            "feature_count",
            "crs",
            "geometry_types",
            "column_count",
        ]
    ]
)

print("\nAquifers with non-standard column count:")
display(
    aquifer_structure[
        aquifer_structure["column_count"] != 8
    ][
        [
            "chapter",
            "filename",
            "feature_count",
            "crs",
            "geometry_types",
            "column_count",
        ]
    ]
)

CRS distribution:


,crs,file_count
0,EPSG:26910,22
1,EPSG:4269,3



Column-count distribution:


,column_count,file_count
0,8,21
1,9,4



Aquifers with multiple features:


,chapter,filename,feature_count,crs,geometry_types,column_count
17,CH7_BLUESKY Maps and shapefiles,GBCS_BLUESKY_AQUIFER_CO2_STORAGE_CHINCHAGA_DAH...,3,EPSG:26910,[Polygon],8
20,CH8_CADOMIN-GETHING Maps and shapefiles,GBCS_CADOMIN_AQUIFER_STODDART_WEST_83UTM10_PG.shp,3,EPSG:4269,[Polygon],8



Aquifers with non-standard column count:


,chapter,filename,feature_count,crs,geometry_types,column_count
4,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_P...,1,EPSG:26910,[MultiPolygon],9
5,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp,1,EPSG:26910,[MultiPolygon],9
6,CH12_HALFWAY Maps and shapefiles,GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp,1,EPSG:26910,[Polygon],9
15,CH16_MID-DEVONIAN CARBONATES Maps and shapefiles,GBCS_SLVPT_AQUIFER_SOUTH_MUSKEG_10UTM83_PG.shp,1,EPSG:26910,[Polygon],9


In [47]:
# ---------------------------------------------------------------------------
# Aquifers using geographic NAD83 rather than UTM 10N
# ---------------------------------------------------------------------------

display(
    aquifer_structure[
        aquifer_structure["crs"] == "EPSG:4269"
    ][
        [
            "chapter",
            "filename",
            "feature_count",
            "geometry_types",
            "column_count",
        ]
    ]
)

,chapter,filename,feature_count,geometry_types,column_count
13,CH16_MID-DEVONIAN CARBONATES Maps and shapefiles,GBCS_SLVPT_AQUIFER_NORTH_10UTM83_PG.shp,1,[Polygon],8
20,CH8_CADOMIN-GETHING Maps and shapefiles,GBCS_CADOMIN_AQUIFER_STODDART_WEST_83UTM10_PG.shp,3,[Polygon],8
21,CH8_CADOMIN-GETHING Maps and shapefiles,GBCS_CADOMIN_AQUIFER_SUNRISE_DOE_83UTM10_PG.shp,1,[Polygon],8


In [48]:
# ---------------------------------------------------------------------------
# Inspect aquifer schemas and feature-level attribute consistency
# ---------------------------------------------------------------------------

aquifer_schema_records = []
multi_feature_checks = []

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]
    gdf = gpd.read_file(shp_path)

    # Record schema
    aquifer_schema_records.append(
        {
            "filename": row["filename"],
            "chapter": row["chapter"],
            "feature_count": len(gdf),
            "crs": gdf.crs.to_string() if gdf.crs else None,
            "columns": list(gdf.columns),
        }
    )

    # Check whether attributes vary across multi-feature aquifers
    if len(gdf) > 1:
        non_geom_cols = [
            col for col in gdf.columns
            if col != gdf.geometry.name
        ]

        varying_fields = [
            col
            for col in non_geom_cols
            if gdf[col].nunique(dropna=False) > 1
        ]

        multi_feature_checks.append(
            {
                "filename": row["filename"],
                "feature_count": len(gdf),
                "varying_field_count": len(varying_fields),
                "varying_fields": varying_fields,
            }
        )

aquifer_schema_records = pd.DataFrame(aquifer_schema_records)
multi_feature_checks = pd.DataFrame(multi_feature_checks)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


In [53]:
# ---------------------------------------------------------------------------
# Diagnose aquifer geometry warnings and validity
# ---------------------------------------------------------------------------

aquifer_geometry_diagnostics = []

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]

    try:
        gdf = gpd.read_file(shp_path)

        aquifer_geometry_diagnostics.append(
            {
                "filename": row["filename"],
                "feature_count": len(gdf),
                "crs": gdf.crs.to_string() if gdf.crs else None,
                "geometry_types": sorted(
                    gdf.geometry.geom_type.dropna().unique().tolist()
                ),
                "valid_geometries": int(gdf.geometry.is_valid.sum()),
                "invalid_geometries": int((~gdf.geometry.is_valid).sum()),
                "null_geometries": int(gdf.geometry.isna().sum()),
                "empty_geometries": int(gdf.geometry.is_empty.sum()),
            }
        )

    except Exception as exc:
        aquifer_geometry_diagnostics.append(
            {
                "filename": row["filename"],
                "feature_count": None,
                "crs": None,
                "geometry_types": None,
                "valid_geometries": None,
                "invalid_geometries": None,
                "null_geometries": None,
                "empty_geometries": None,
                "error": str(exc),
            }
        )

aquifer_geometry_diagnostics = pd.DataFrame(
    aquifer_geometry_diagnostics
)

display(aquifer_geometry_diagnostics)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,filename,feature_count,crs,geometry_types,valid_geometries,invalid_geometries,null_geometries,empty_geometries
0,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,1,EPSG:26910,[Polygon],1,0,0,0
1,GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp,1,EPSG:26910,[Polygon],1,0,0,0
2,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,1,EPSG:26910,[MultiPolygon],0,1,0,0
3,GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM...,1,EPSG:26910,[Polygon],1,0,0,0
4,GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_P...,1,EPSG:26910,[MultiPolygon],0,1,0,0
5,GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp,1,EPSG:26910,[MultiPolygon],1,0,0,0
6,GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp,1,EPSG:26910,[Polygon],1,0,0,0
7,GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp,1,EPSG:26910,[Polygon],1,0,0,0
8,GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp,1,EPSG:26910,[Polygon],1,0,0,0
9,GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp,1,EPSG:26910,[Polygon],1,0,0,0


In [54]:
problem_aquifers = aquifer_geometry_diagnostics[
    (aquifer_geometry_diagnostics["invalid_geometries"] > 0)
    | (aquifer_geometry_diagnostics["null_geometries"] > 0)
    | (aquifer_geometry_diagnostics["empty_geometries"] > 0)
]

display(problem_aquifers)

,filename,feature_count,crs,geometry_types,valid_geometries,invalid_geometries,null_geometries,empty_geometries
2,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,1,EPSG:26910,[MultiPolygon],0,1,0,0
4,GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_P...,1,EPSG:26910,[MultiPolygon],0,1,0,0


In [55]:
# ---------------------------------------------------------------------------
# Display multi-feature aquifer records
# ---------------------------------------------------------------------------

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]

    print(f"Reading: {row['filename']}")

    gdf = gpd.read_file(shp_path)

    if len(gdf) > 1:
        print("-" * 100)
        display(gdf)

Reading: GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83UTM10_PG.shp
Reading: GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp
Reading: GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp
Reading: GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM10_PG.shp
Reading: GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_PG.shp
Reading: GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp
Reading: GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp
Reading: GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp
Reading: GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp
Reading: GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp
Reading: GBCS_DEBOLT_AQUIFER_NORTH_OSBORN_RING_83UTM83_PG.shp
Reading: GBCS_DEBOLT_AQUIFER_WEST_BLUEBERRY_83UTM83_PG.shp
Reading: GBCS_SLVPT_AQUIFER_CENTRAL_10UTM83_PG.shp
Reading: GBCS_SLVPT_AQUIFER_NORTH_10UTM83_PG.shp
Reading: GBCS_SLVPT_AQUIFER_SOUTH_10UTM83_PG.shp
Reading: GBCS_SLVPT_AQUIFER_SOUTH_MUSKEG_10UTM83_PG.shp
Reading: GBCS_PEACE_RIVER_AQUIFER_DAWSON_83UTM10_PG.shp
Reading: GBCS_BLUESKY

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,Th_SCap_MT,Eff_SCap_h,Eff_Scap_2,Eff_Scap_5,CO2_PHASE,SHAPE_Leng,SHAPE_Area,geometry
0,304.0,1.5,6.1,16.4,Gas,273300.231223,2.283830e+09,"POLYGON ((681209.825 6325324.056, 678982.976 6..."
1,0.0,0.0,0.0,0.0,Supercritical,35122.121530,9.494170e+07,"POLYGON ((609777.064 6347771.054, 609909.74 63..."
2,304.0,1.5,6.1,16.4,Supercritical,93271.766838,4.842147e+08,"POLYGON ((681209.825 6325324.056, 682068.923 6..."


Reading: GBCS_BLUESKY_AQUIFER_CO2_STORAGE_DOE_AIRPORT_83UTM10_PG.shp
Reading: GBCS_CADOMIN_AQUIFER_PARKLAND_MUSKRAT_83UTM10_PG.shp
Reading: GBCS_CADOMIN_AQUIFER_STODDART_WEST_83UTM10_PG.shp
----------------------------------------------------------------------------------------------------


,Th_SCap_MT,Eff_SCap_h,Eff_Scap_2,Eff_Scap_5,CO2_PHASE,SHAPE_Leng,SHAPE_Area,geometry
0,3248.9,16.2,65.0,175.4,Gas,2.112736,0.181884,"POLYGON ((-121.591 56.28772, -121.57617 56.275..."
1,3248.9,16.2,65.0,175.4,Supercritcal,1.892341,0.085602,"POLYGON ((-121.5943 55.9447, -121.58478 55.946..."
2,3248.9,16.2,65.0,175.4,Supercritcal,2.023208,0.194887,"POLYGON ((-121.30847 56.60292, -121.29242 56.5..."


Reading: GBCS_CADOMIN_AQUIFER_SUNRISE_DOE_83UTM10_PG.shp
Reading: GBCS_NIKANASSIN_AQUIFER_BEG_SIPHON_83UTM10_PG.shp
Reading: GBCS_NIKANASSIN_AQUIFER_BLUEBERRY_TWO_RIVERS_83UTM10_PG.shp
Reading: GBCS_NIKANASSIN_AQUIFER_BRASSEY_CUTBANK_83UTM10_PG.shp


In [49]:
# ---------------------------------------------------------------------------
# Distinct aquifer field schemas
# ---------------------------------------------------------------------------

schema_summary = (
    aquifer_schema_records["columns"]
    .apply(tuple)
    .value_counts()
    .rename_axis("schema")
    .reset_index(name="file_count")
)

display(schema_summary)

,schema,file_count
0,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, Eff_Scap_...",10
1,"(TH_SCap_MT, Eff_SCap_h, Eff_SCap_2, Eff_SCap_...",7
2,"(NAME, Th_SCap_MT, Eff_SCap_h, Eff_SCap_2, Eff...",3
3,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, Eff_Scap_...",2
4,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, Eff_Scap_...",1
5,"(REC_ID, Th_SCap_MT, Eff_SCap_h, Eff_Scap_2, E...",1
6,"(Th_SCap_MT, Eff_SCap_h, Eff_Scap_5, Eff_Scap_...",1


In [56]:
# ---------------------------------------------------------------------------
# Compare distinct aquifer field schemas
# ---------------------------------------------------------------------------

schema_summary = (
    aquifer_schema_records["columns"]
    .apply(tuple)
    .value_counts()
    .rename_axis("schema")
    .reset_index(name="file_count")
)

print(f"Distinct aquifer schemas: {len(schema_summary)}\n")

for i, row in schema_summary.iterrows():
    print("=" * 80)
    print(f"Schema {i + 1} — used by {row['file_count']} file(s)")
    print("-" * 80)

    for col in row["schema"]:
        print(col)

    print()

Distinct aquifer schemas: 7

Schema 1 — used by 10 file(s)
--------------------------------------------------------------------------------
Th_SCap_MT
Eff_SCap_h
Eff_Scap_2
Eff_Scap_5
CO2_PHASE
SHAPE_Leng
SHAPE_Area
geometry

Schema 2 — used by 7 file(s)
--------------------------------------------------------------------------------
TH_SCap_MT
Eff_SCap_h
Eff_SCap_2
Eff_SCap_5
CO2_Phase
SHAPE_Leng
SHAPE_Area
geometry

Schema 3 — used by 3 file(s)
--------------------------------------------------------------------------------
NAME
Th_SCap_MT
Eff_SCap_h
Eff_SCap_2
Eff_SCap_5
CO2_PHASE
SHAPE_Leng
SHAPE_Area
geometry

Schema 4 — used by 2 file(s)
--------------------------------------------------------------------------------
Th_SCap_MT
Eff_SCap_h
Eff_Scap_2
Eff_Scap_5
CO2_PHASE
Shape_Leng
Shape_Area
geometry

Schema 5 — used by 1 file(s)
--------------------------------------------------------------------------------
Th_SCap_MT
Eff_SCap_h
Eff_Scap_2
Eff_Scap_5
CO2_Phase
Shape_Leng
Shape_

In [57]:
# ---------------------------------------------------------------------------
# Identify files associated with each aquifer schema
# ---------------------------------------------------------------------------

aquifer_schema_records["schema"] = (
    aquifer_schema_records["columns"]
    .apply(tuple)
)

for i, schema in enumerate(schema_summary["schema"], start=1):
    matching = aquifer_schema_records[
        aquifer_schema_records["schema"] == schema
    ]

    print("=" * 80)
    print(f"Schema {i}")
    print("-" * 80)

    for filename in matching["filename"]:
        print(filename)

    print()

Schema 1
--------------------------------------------------------------------------------
GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp
GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp
GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp
GBCS_DEBOLT_AQUIFER_NORTH_OSBORN_RING_83UTM83_PG.shp
GBCS_DEBOLT_AQUIFER_WEST_BLUEBERRY_83UTM83_PG.shp
GBCS_PEACE_RIVER_AQUIFER_DAWSON_83UTM10_PG.shp
GBCS_BLUESKY_AQUIFER_CO2_STORAGE_CHINCHAGA_DAHL_83UTM10_PG.shp
GBCS_BLUESKY_AQUIFER_CO2_STORAGE_DOE_AIRPORT_83UTM10_PG.shp
GBCS_CADOMIN_AQUIFER_STODDART_WEST_83UTM10_PG.shp
GBCS_CADOMIN_AQUIFER_SUNRISE_DOE_83UTM10_PG.shp

Schema 2
--------------------------------------------------------------------------------
GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83UTM10_PG.shp
GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp
GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp
GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM10_PG.shp
GBCS_NIKANASSIN_AQUIFER_BEG_SIPHON_83UTM10_PG.shp
GBCS_NIKANASSIN_AQUIFER_BLUEBERRY_TWO_RIVER

In [58]:
# ---------------------------------------------------------------------------
# Inspect source-specific aquifer identifier fields
# ---------------------------------------------------------------------------

identifier_fields = ["NAME", "REC_ID"]

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]
    gdf = gpd.read_file(shp_path)

    present_fields = [
        field
        for field in identifier_fields
        if field in gdf.columns
    ]

    if not present_fields:
        continue

    print("\n" + "=" * 100)
    print(row["filename"])
    print("-" * 100)

    display(
        gdf[
            present_fields
            + [
                col
                for col in [
                    "Th_SCap_MT",
                    "TH_SCap_MT",
                    "Eff_SCap_h",
                    "Eff_SCap_2",
                    "Eff_Scap_2",
                    "Eff_SCap_5",
                    "Eff_Scap_5",
                    "CO2_PHASE",
                    "CO2_Phase",
                ]
                if col in gdf.columns
            ]
        ]
    )


GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_PG.shp
----------------------------------------------------------------------------------------------------


c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,NAME,Th_SCap_MT,Eff_SCap_h,Eff_SCap_2,Eff_SCap_5,CO2_PHASE
0,Flatrock Monias Regional System,2316.891605,11.5,46.3,125.1,Supercritical



GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp
----------------------------------------------------------------------------------------------------


,NAME,Th_SCap_MT,Eff_SCap_h,Eff_SCap_2,Eff_SCap_5,CO2_PHASE
0,Peejay-Weasel Regional System (East),1514.785508,7.6,30.3,81.8,Supercritical



GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp
----------------------------------------------------------------------------------------------------


,NAME,Th_SCap_MT,Eff_SCap_h,Eff_SCap_2,Eff_SCap_5,CO2_PHASE
0,Fireweed-Martin Regional Syste West),1125.933804,5.6,22.5,60.8,Supercritical



GBCS_SLVPT_AQUIFER_SOUTH_MUSKEG_10UTM83_PG.shp
----------------------------------------------------------------------------------------------------


,REC_ID,Th_SCap_MT,Eff_SCap_h,Eff_Scap_2,Eff_Scap_5,CO2_Phase
0,0.0,12394.9491,62.0,247.9,669.3,Supercritical


In [59]:
# ---------------------------------------------------------------------------
# Document equivalent aquifer source fields
# ---------------------------------------------------------------------------

aquifer_field_equivalents = {
    "theoretical_storage_mt": [
        "Th_SCap_MT",
        "TH_SCap_MT",
    ],
    "effective_storage_low_mt": [
        "Eff_SCap_h",
    ],
    "effective_storage_mid_mt": [
        "Eff_SCap_2",
        "Eff_Scap_2",
    ],
    "effective_storage_high_mt": [
        "Eff_SCap_5",
        "Eff_Scap_5",
    ],
    "co2_phase": [
        "CO2_PHASE",
        "CO2_Phase",
    ],
    "shape_length": [
        "SHAPE_Leng",
        "Shape_Leng",
    ],
    "shape_area": [
        "SHAPE_Area",
        "Shape_Area",
    ],
}

for canonical, variants in aquifer_field_equivalents.items():
    print(f"{canonical}:")
    for variant in variants:
        print(f"  - {variant}")

theoretical_storage_mt:
  - Th_SCap_MT
  - TH_SCap_MT
effective_storage_low_mt:
  - Eff_SCap_h
effective_storage_mid_mt:
  - Eff_SCap_2
  - Eff_Scap_2
effective_storage_high_mt:
  - Eff_SCap_5
  - Eff_Scap_5
co2_phase:
  - CO2_PHASE
  - CO2_Phase
shape_length:
  - SHAPE_Leng
  - Shape_Leng
shape_area:
  - SHAPE_Area
  - Shape_Area


In [61]:
# ---------------------------------------------------------------------------
# Map aquifer shapefiles to Appendix C aquifer names
# ---------------------------------------------------------------------------

aquifer_name_map = {
    "GBCS_PEACE_RIVER_AQUIFER_DAWSON_83UTM10_PG.shp":
        "Dawson Creek",

    "GBCS_BLUESKY_AQUIFER_CO2_STORAGE_CHINCHAGA_DAHL_83UTM10_PG.shp":
        "Chinchaga-Dahl",

    "GBCS_BLUESKY_AQUIFER_CO2_STORAGE_DOE_AIRPORT_83UTM10_PG.shp":
        "Doe-Airport",

    "GBCS_CADOMIN_AQUIFER_PARKLAND_MUSKRAT_83UTM10_PG.shp":
        "Parkland-Muskrat",

    "GBCS_CADOMIN_AQUIFER_STODDART_WEST_83UTM10_PG.shp":
        "Stoddart West",

    "GBCS_CADOMIN_AQUIFER_SUNRISE_DOE_83UTM10_PG.shp":
        "Sunrise-Doe",

    "GBCS_NIKANASSIN_AQUIFER_BEG_SIPHON_83UTM10_PG.shp":
        "Beg-Siphon",

    "GBCS_NIKANASSIN_AQUIFER_BLUEBERRY_TWO_RIVERS_83UTM10_PG.shp":
        "Blueberry-Two Rivers",

    "GBCS_NIKANASSIN_AQUIFER_BRASSEY_CUTBANK_83UTM10_PG.shp":
        "Brassey-Cutbank",

    "GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83UTM10_PG.shp":
        "Boundary Lake-Osprey",

    "GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp":
        "Dawson",

    "GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp":
        "Monias-Beg",

    "GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM10_PG.shp":
        "Parkland-Stoddart",

    "GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_PG.shp":
        "Flatrock-Monias",

    "GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp":
        "Peejay-Weasel",

    "GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp":
        "Fireweed-Martin",

    "GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp":
        "Boundary Lake",

    "GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp":
        "Doe-Stoddart",

    "GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp":
        "Ladyfern-Ring",

    "GBCS_DEBOLT_AQUIFER_NORTH_OSBORN_RING_83UTM83_PG.shp":
        "Osborn-Ring",

    "GBCS_DEBOLT_AQUIFER_WEST_BLUEBERRY_83UTM83_PG.shp":
        "Blueberry-Buick",

    "GBCS_SLVPT_AQUIFER_NORTH_10UTM83_PG.shp":
        "North",

    "GBCS_SLVPT_AQUIFER_CENTRAL_10UTM83_PG.shp":
        "Central",

    "GBCS_SLVPT_AQUIFER_SOUTH_10UTM83_PG.shp":
        "South",

    "GBCS_SLVPT_AQUIFER_SOUTH_MUSKEG_10UTM83_PG.shp":
        "South Muskeg",
}

print(
    "Mapped aquifer shapefiles:",
    len(aquifer_name_map),
)

Mapped aquifer shapefiles: 25


In [62]:
# ---------------------------------------------------------------------------
# Build one logical record per aquifer shapefile
# ---------------------------------------------------------------------------

def first_existing_value(gdf, candidates):
    for field in candidates:
        if field in gdf.columns:
            return gdf[field].iloc[0]
    return None


aquifer_shp_records = []

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]
    gdf = gpd.read_file(shp_path)

    aquifer_shp_records.append(
        {
            "filename": row["filename"],
            "aquifer_name": aquifer_name_map[row["filename"]],
            "feature_count": len(gdf),

            "theoretical_storage_mt": first_existing_value(
                gdf,
                ["Th_SCap_MT", "TH_SCap_MT"],
            ),

            "p10_mt": first_existing_value(
                gdf,
                ["Eff_SCap_h"],
            ),

            "p50_mt": first_existing_value(
                gdf,
                ["Eff_SCap_2", "Eff_Scap_2"],
            ),

            "p90_mt": first_existing_value(
                gdf,
                ["Eff_SCap_5", "Eff_Scap_5"],
            ),

            "co2_phase": first_existing_value(
                gdf,
                ["CO2_PHASE", "CO2_Phase"],
            ),
        }
    )

aquifer_shp_summary = pd.DataFrame(aquifer_shp_records)

print(f"Aquifer SHP logical records: {len(aquifer_shp_summary)}")

display(aquifer_shp_summary)

Aquifer SHP logical records: 25


c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,filename,aquifer_name,feature_count,theoretical_storage_mt,p10_mt,p50_mt,p90_mt,co2_phase
0,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,Boundary Lake-Osprey,1,0.000000,3.4,13.600000,36.80000,Supercritical
1,GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp,Dawson,1,0.000000,18.0,71.600000,193.40000,Supercritical
2,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,Monias-Beg,1,0.000000,9.2,36.700000,99.00000,Supercritical
3,GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM...,Parkland-Stoddart,1,0.000000,15.6,62.300000,168.20000,Supercritical
4,GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_P...,Flatrock-Monias,1,2316.891605,11.5,46.300000,125.10000,Supercritical
5,GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp,Peejay-Weasel,1,1514.785508,7.6,30.300000,81.80000,Supercritical
6,GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp,Fireweed-Martin,1,1125.933804,5.6,22.500000,60.80000,Supercritical
7,GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp,Boundary Lake,1,2433.219081,12.166095,48.664382,131.39383,Supercritical
8,GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp,Doe-Stoddart,1,0.000000,24.9,99.600000,268.80000,Supercitical
9,GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp,Ladyfern-Ring,1,507.318832,2.5,10.100000,27.40000,Supercritical


In [63]:
# ---------------------------------------------------------------------------
# Verify attribute consistency within multi-feature aquifers
# ---------------------------------------------------------------------------

multi_feature_attribute_checks = []

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]
    gdf = gpd.read_file(shp_path)

    if len(gdf) <= 1:
        continue

    field_groups = {
        "theoretical_storage_mt": ["Th_SCap_MT", "TH_SCap_MT"],
        "p10_mt": ["Eff_SCap_h"],
        "p50_mt": ["Eff_SCap_2", "Eff_Scap_2"],
        "p90_mt": ["Eff_SCap_5", "Eff_Scap_5"],
        "co2_phase": ["CO2_PHASE", "CO2_Phase"],
    }

    result = {
        "filename": row["filename"],
        "feature_count": len(gdf),
    }

    for canonical, candidates in field_groups.items():
        source_field = next(
            (field for field in candidates if field in gdf.columns),
            None,
        )

        if source_field is None:
            result[f"{canonical}_unique"] = None
        else:
            result[f"{canonical}_unique"] = (
                gdf[source_field].nunique(dropna=False)
            )

    multi_feature_attribute_checks.append(result)

multi_feature_attribute_checks = pd.DataFrame(
    multi_feature_attribute_checks
)

display(multi_feature_attribute_checks)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,filename,feature_count,theoretical_storage_mt_unique,p10_mt_unique,p50_mt_unique,p90_mt_unique,co2_phase_unique
0,GBCS_BLUESKY_AQUIFER_CO2_STORAGE_CHINCHAGA_DAH...,3,2,2,2,2,2
1,GBCS_CADOMIN_AQUIFER_STODDART_WEST_83UTM10_PG.shp,3,1,1,1,1,2


In [64]:
# ---------------------------------------------------------------------------
# Inspect attribute differences within multi-feature aquifers
# ---------------------------------------------------------------------------

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]
    gdf = gpd.read_file(shp_path)

    if len(gdf) <= 1:
        continue

    print("\n" + "=" * 100)
    print(row["filename"])
    print("-" * 100)

    comparison = pd.DataFrame(
        {
            "feature_id": gdf.index,
            "theoretical_storage_mt": [
                first_existing_value(
                    gdf.iloc[[i]],
                    ["Th_SCap_MT", "TH_SCap_MT"],
                )
                for i in range(len(gdf))
            ],
            "p10_mt": [
                first_existing_value(
                    gdf.iloc[[i]],
                    ["Eff_SCap_h"],
                )
                for i in range(len(gdf))
            ],
            "p50_mt": [
                first_existing_value(
                    gdf.iloc[[i]],
                    ["Eff_SCap_2", "Eff_Scap_2"],
                )
                for i in range(len(gdf))
            ],
            "p90_mt": [
                first_existing_value(
                    gdf.iloc[[i]],
                    ["Eff_SCap_5", "Eff_Scap_5"],
                )
                for i in range(len(gdf))
            ],
            "co2_phase": [
                first_existing_value(
                    gdf.iloc[[i]],
                    ["CO2_PHASE", "CO2_Phase"],
                )
                for i in range(len(gdf))
            ],
        }
    )

    display(comparison)


GBCS_BLUESKY_AQUIFER_CO2_STORAGE_CHINCHAGA_DAHL_83UTM10_PG.shp
----------------------------------------------------------------------------------------------------


c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,feature_id,theoretical_storage_mt,p10_mt,p50_mt,p90_mt,co2_phase
0,0,304.0,1.5,6.1,16.4,Gas
1,1,0.0,0.0,0.0,0.0,Supercritical
2,2,304.0,1.5,6.1,16.4,Supercritical



GBCS_CADOMIN_AQUIFER_STODDART_WEST_83UTM10_PG.shp
----------------------------------------------------------------------------------------------------


,feature_id,theoretical_storage_mt,p10_mt,p50_mt,p90_mt,co2_phase
0,0,3248.9,16.2,65.0,175.4,Gas
1,1,3248.9,16.2,65.0,175.4,Supercritcal
2,2,3248.9,16.2,65.0,175.4,Supercritcal


In [65]:
# ---------------------------------------------------------------------------
# Compare logical aquifer storage values to Appendix C
# without summing repeated spatial features
# ---------------------------------------------------------------------------

aquifer_shp_records = []

for _, row in aquifer_shapefiles.iterrows():
    shp_path = APPENDIX_E / row["relative_path"]
    gdf = gpd.read_file(shp_path)

    def unique_nonzero(candidates):
        for field in candidates:
            if field in gdf.columns:
                values = pd.to_numeric(
                    gdf[field],
                    errors="coerce"
                ).dropna()

                nonzero = values[values != 0].unique()

                if len(nonzero) == 1:
                    return nonzero[0]

                if len(nonzero) == 0:
                    zero_values = values.unique()
                    return zero_values[0] if len(zero_values) == 1 else None

                return list(nonzero)

        return None

    phase_field = next(
        (
            field for field in ["CO2_PHASE", "CO2_Phase"]
            if field in gdf.columns
        ),
        None,
    )

    phases = (
        sorted(
            gdf[phase_field]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        if phase_field
        else []
    )

    aquifer_shp_records.append(
        {
            "filename": row["filename"],
            "aquifer_name": aquifer_name_map[row["filename"]],
            "feature_count": len(gdf),

            "theoretical_storage_mt": unique_nonzero(
                ["Th_SCap_MT", "TH_SCap_MT"]
            ),

            "p10_mt": unique_nonzero(
                ["Eff_SCap_h"]
            ),

            "p50_mt": unique_nonzero(
                ["Eff_SCap_2", "Eff_Scap_2"]
            ),

            "p90_mt": unique_nonzero(
                ["Eff_SCap_5", "Eff_Scap_5"]
            ),

            "co2_phases": phases,
        }
    )

aquifer_shp_summary = pd.DataFrame(aquifer_shp_records)

display(aquifer_shp_summary)

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 0 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


,filename,aquifer_name,feature_count,theoretical_storage_mt,p10_mt,p50_mt,p90_mt,co2_phases
0,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,Boundary Lake-Osprey,1,0.000000,3.400000,13.600000,36.80000,[Supercritical]
1,GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp,Dawson,1,0.000000,18.000000,71.600000,193.40000,[Supercritical]
2,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,Monias-Beg,1,0.000000,9.200000,36.700000,99.00000,[Supercritical]
3,GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM...,Parkland-Stoddart,1,0.000000,15.600000,62.300000,168.20000,[Supercritical]
4,GBCS_HALFWAY_AQUIFER_FLATROCK_MONIAS_83UTM10_P...,Flatrock-Monias,1,2316.891605,11.500000,46.300000,125.10000,[Supercritical]
5,GBCS_HALFWAY_AQUIFER_PEEJAY_WEASEL_83UTM10_PG.shp,Peejay-Weasel,1,1514.785508,7.600000,30.300000,81.80000,[Supercritical]
6,GBCS_HALFWAY_AQUIFER_WEST_FIREWEED_83UTM10_PG.shp,Fireweed-Martin,1,1125.933804,5.600000,22.500000,60.80000,[Supercritical]
7,GBCS_BELLOY_AQUIFER_BOUNDARY_LAKE_83UTM10_PG.shp,Boundary Lake,1,2433.219081,12.166095,48.664382,131.39383,[Supercritical]
8,GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp,Doe-Stoddart,1,0.000000,24.900000,99.600000,268.80000,[Supercitical]
9,GBCS_BELLOY_AQUIFER_LADYFERN_RING_83UTM10_PG.shp,Ladyfern-Ring,1,507.318832,2.500000,10.100000,27.40000,[Supercritical]


In [66]:
# ---------------------------------------------------------------------------
# Join aquifer SHP summary to Appendix C
# ---------------------------------------------------------------------------

aquifer_comparison = aquifer_shp_summary.merge(
    aquifers,
    left_on="aquifer_name",
    right_on="Aquifer Name",
    how="left",
    validate="one_to_one",
)

print(f"Comparison rows: {len(aquifer_comparison)}")
print(
    "Missing Appendix C matches:",
    aquifer_comparison["Aquifer Name"].isna().sum()
)

Comparison rows: 25
Missing Appendix C matches: 1


In [67]:
# ---------------------------------------------------------------------------
# Compare SHP storage values against Appendix C
# ---------------------------------------------------------------------------

comparison_fields = {
    "p10_mt": "P10 Effective Storage Potential at 0.5% (Mt)",
    "p50_mt": "P50 Effective Storage Potential at 2% (Mt)",
    "p90_mt": "P90 Effective Storage Potential at 5.4% (Mt)",
}

results = []

for shp_field, xlsx_field in comparison_fields.items():
    left = pd.to_numeric(
        aquifer_comparison[shp_field],
        errors="coerce",
    )

    right = pd.to_numeric(
        aquifer_comparison[xlsx_field],
        errors="coerce",
    )

    match = (
        (left - right).abs() < 0.11
    ) | (
        left.isna() & right.isna()
    )

    results.append(
        {
            "shapefile_field": shp_field,
            "appendix_c_field": xlsx_field,
            "matches": int(match.sum()),
            "rows": len(match),
            "match_pct": 100 * match.mean(),
        }
    )

aquifer_value_summary = pd.DataFrame(results)

display(aquifer_value_summary)

,shapefile_field,appendix_c_field,matches,rows,match_pct
0,p10_mt,P10 Effective Storage Potential at 0.5% (Mt),21,25,84.0
1,p50_mt,P50 Effective Storage Potential at 2% (Mt),21,25,84.0
2,p90_mt,P90 Effective Storage Potential at 5.4% (Mt),21,25,84.0


In [68]:
# ---------------------------------------------------------------------------
# Diagnose aquifer join and storage-value mismatches
# ---------------------------------------------------------------------------

# 1. Missing Appendix C name match
missing_aquifer_matches = aquifer_comparison[
    aquifer_comparison["Aquifer Name"].isna()
][
    [
        "filename",
        "aquifer_name",
        "feature_count",
        "p10_mt",
        "p50_mt",
        "p90_mt",
    ]
]

print("Missing Appendix C matches:")
display(missing_aquifer_matches)


# 2. Compare storage values row-by-row
comparison_fields = {
    "p10_mt": "P10 Effective Storage Potential at 0.5% (Mt)",
    "p50_mt": "P50 Effective Storage Potential at 2% (Mt)",
    "p90_mt": "P90 Effective Storage Potential at 5.4% (Mt)",
}

mismatch_masks = []

for shp_field, xlsx_field in comparison_fields.items():
    left = pd.to_numeric(
        aquifer_comparison[shp_field],
        errors="coerce",
    )

    right = pd.to_numeric(
        aquifer_comparison[xlsx_field],
        errors="coerce",
    )

    match = (
        (left - right).abs() < 0.11
    ) | (
        left.isna() & right.isna()
    )

    aquifer_comparison[f"{shp_field}_match"] = match
    mismatch_masks.append(~match)


# Any aquifer that mismatches at least one storage estimate
any_mismatch = pd.concat(
    mismatch_masks,
    axis=1,
).any(axis=1)

storage_mismatches = aquifer_comparison.loc[
    any_mismatch,
    [
        "filename",
        "aquifer_name",
        "Aquifer Name",
        "feature_count",

        "p10_mt",
        "P10 Effective Storage Potential at 0.5% (Mt)",
        "p10_mt_match",

        "p50_mt",
        "P50 Effective Storage Potential at 2% (Mt)",
        "p50_mt_match",

        "p90_mt",
        "P90 Effective Storage Potential at 5.4% (Mt)",
        "p90_mt_match",
    ]
]

print(f"\nAquifers with storage mismatches: {len(storage_mismatches)}")
display(storage_mismatches)

Missing Appendix C matches:


,filename,aquifer_name,feature_count,p10_mt,p50_mt,p90_mt
11,GBCS_DEBOLT_AQUIFER_WEST_BLUEBERRY_83UTM83_PG.shp,Blueberry-Buick,1,7.7,30.7,82.8



Aquifers with storage mismatches: 4


,filename,aquifer_name,Aquifer Name,feature_count,p10_mt,P10 Effective Storage Potential at 0.5% (Mt),p10_mt_match,p50_mt,P50 Effective Storage Potential at 2% (Mt),p50_mt_match,p90_mt,P90 Effective Storage Potential at 5.4% (Mt),p90_mt_match
11,GBCS_DEBOLT_AQUIFER_WEST_BLUEBERRY_83UTM83_PG.shp,Blueberry-Buick,NaN,1,7.7,NaN,False,30.7,NaN,False,82.8,NaN,False
22,GBCS_NIKANASSIN_AQUIFER_BEG_SIPHON_83UTM10_PG.shp,Beg-Siphon,Beg-Siphon,1,6.6,5.934243,False,26.3,23.736970,False,70.9,64.089819,False
23,GBCS_NIKANASSIN_AQUIFER_BLUEBERRY_TWO_RIVERS_8...,Blueberry-Two Rivers,Blueberry-Two Rivers,1,112.9,110.150721,False,451.7,440.602886,False,1219.7,1189.627792,False
24,GBCS_NIKANASSIN_AQUIFER_BRASSEY_CUTBANK_83UTM1...,Brassey-Cutbank,Brassey-Cutbank,1,33.2,23.928082,False,133.0,95.712327,False,359.0,258.423282,False


In [69]:
# ---------------------------------------------------------------------------
# Quantify SHP ↔ Appendix C aquifer storage differences
# ---------------------------------------------------------------------------

numeric_mismatches = storage_mismatches[
    storage_mismatches["Aquifer Name"].notna()
].copy()

for percentile in ["p10", "p50", "p90"]:
    shp_col = f"{percentile}_mt"

    appendix_col = {
        "p10": "P10 Effective Storage Potential at 0.5% (Mt)",
        "p50": "P50 Effective Storage Potential at 2% (Mt)",
        "p90": "P90 Effective Storage Potential at 5.4% (Mt)",
    }[percentile]

    numeric_mismatches[f"{percentile}_difference_mt"] = (
        numeric_mismatches[shp_col]
        - numeric_mismatches[appendix_col]
    )

    numeric_mismatches[f"{percentile}_ratio"] = (
        numeric_mismatches[shp_col]
        / numeric_mismatches[appendix_col]
    )

display(
    numeric_mismatches[
        [
            "aquifer_name",
            "p10_mt",
            "P10 Effective Storage Potential at 0.5% (Mt)",
            "p10_ratio",
            "p50_mt",
            "P50 Effective Storage Potential at 2% (Mt)",
            "p50_ratio",
            "p90_mt",
            "P90 Effective Storage Potential at 5.4% (Mt)",
            "p90_ratio",
        ]
    ]
)

,aquifer_name,p10_mt,P10 Effective Storage Potential at 0.5% (Mt),p10_ratio,p50_mt,P50 Effective Storage Potential at 2% (Mt),p50_ratio,p90_mt,P90 Effective Storage Potential at 5.4% (Mt),p90_ratio
22,Beg-Siphon,6.6,5.934243,1.112189,26.3,23.736970,1.107976,70.9,64.089819,1.106260
23,Blueberry-Two Rivers,112.9,110.150721,1.024959,451.7,440.602886,1.025186,1219.7,1189.627792,1.025279
24,Brassey-Cutbank,33.2,23.928082,1.387491,133.0,95.712327,1.389581,359.0,258.423282,1.389194


In [71]:
# ---------------------------------------------------------------------------
# Compare SHP theoretical storage to Appendix C implied theoretical storage
# ---------------------------------------------------------------------------

theoretical_check = aquifer_comparison.copy()

theoretical_check["appendix_theoretical_from_p10"] = (
    pd.to_numeric(
        theoretical_check[
            "P10 Effective Storage Potential at 0.5% (Mt)"
        ],
        errors="coerce",
    )
    / 0.005
)

theoretical_check["appendix_theoretical_from_p50"] = (
    pd.to_numeric(
        theoretical_check[
            "P50 Effective Storage Potential at 2% (Mt)"
        ],
        errors="coerce",
    )
    / 0.02
)

theoretical_check["appendix_theoretical_from_p90"] = (
    pd.to_numeric(
        theoretical_check[
            "P90 Effective Storage Potential at 5.4% (Mt)"
        ],
        errors="coerce",
    )
    / 0.054
)

theoretical_check["shp_to_appendix_theoretical_ratio"] = (
    pd.to_numeric(
        theoretical_check["theoretical_storage_mt"],
        errors="coerce",
    )
    / theoretical_check["appendix_theoretical_from_p50"]
)

display(
    theoretical_check.loc[
        theoretical_check["aquifer_name"].isin(
            [
                "Beg-Siphon",
                "Blueberry-Two Rivers",
                "Brassey-Cutbank",
            ]
        ),
        [
            "aquifer_name",
            "theoretical_storage_mt",
            "appendix_theoretical_from_p10",
            "appendix_theoretical_from_p50",
            "appendix_theoretical_from_p90",
            "shp_to_appendix_theoretical_ratio",
        ],
    ]
)

,aquifer_name,theoretical_storage_mt,appendix_theoretical_from_p10,appendix_theoretical_from_p50,appendix_theoretical_from_p90,shp_to_appendix_theoretical_ratio
22,Beg-Siphon,0.0,1186.848503,1186.848503,1186.848503,0.0
23,Blueberry-Two Rivers,0.0,22030.144289,22030.144289,22030.144289,0.0
24,Brassey-Cutbank,0.0,4785.616328,4785.616328,4785.616328,0.0


In [72]:
# ---------------------------------------------------------------------------
# Identify aquifers with zero/missing theoretical storage
# but nonzero effective storage
# ---------------------------------------------------------------------------

theoretical_anomalies = aquifer_shp_summary[
    (
        pd.to_numeric(
            aquifer_shp_summary["theoretical_storage_mt"],
            errors="coerce",
        ).fillna(0) == 0
    )
    &
    (
        pd.to_numeric(
            aquifer_shp_summary["p50_mt"],
            errors="coerce",
        ).fillna(0) > 0
    )
].copy()

print(
    "Aquifers with zero theoretical storage "
    "but nonzero P50 storage:",
    len(theoretical_anomalies),
)

display(
    theoretical_anomalies[
        [
            "filename",
            "aquifer_name",
            "feature_count",
            "theoretical_storage_mt",
            "p10_mt",
            "p50_mt",
            "p90_mt",
            "co2_phases",
        ]
    ]
)

Aquifers with zero theoretical storage but nonzero P50 storage: 8


,filename,aquifer_name,feature_count,theoretical_storage_mt,p10_mt,p50_mt,p90_mt,co2_phases
0,GBCS_BALDONNEL_AQUIFER_BOUNDARY_LAKE_OSPREY_83...,Boundary Lake-Osprey,1,0.0,3.4,13.6,36.8,[Supercritical]
1,GBCS_BALDONNEL_AQUIFER_DAWSON_83UTM10_PG.shp,Dawson,1,0.0,18.0,71.6,193.4,[Supercritical]
2,GBCS_BALDONNEL_AQUIFER_MONIAS_BEG_83UTM10_PG.shp,Monias-Beg,1,0.0,9.2,36.7,99.0,[Supercritical]
3,GBCS_BALDONNEL_AQUIFER_PARKLAND_STODDART_83UTM...,Parkland-Stoddart,1,0.0,15.6,62.3,168.2,[Supercritical]
8,GBCS_BELLOY_AQUIFER_DOE_STODDART_83UTM10_PG.shp,Doe-Stoddart,1,0.0,24.9,99.6,268.8,[Supercitical]
22,GBCS_NIKANASSIN_AQUIFER_BEG_SIPHON_83UTM10_PG.shp,Beg-Siphon,1,0.0,6.6,26.3,70.9,[Supercritical]
23,GBCS_NIKANASSIN_AQUIFER_BLUEBERRY_TWO_RIVERS_8...,Blueberry-Two Rivers,1,0.0,112.9,451.7,1219.7,[Supercritical]
24,GBCS_NIKANASSIN_AQUIFER_BRASSEY_CUTBANK_83UTM1...,Brassey-Cutbank,1,0.0,33.2,133.0,359.0,[Supercritical]


In [73]:
# ---------------------------------------------------------------------------
# Test internal SHP theoretical → effective storage consistency
# ---------------------------------------------------------------------------

internal_check = aquifer_shp_summary.copy()

theoretical = pd.to_numeric(
    internal_check["theoretical_storage_mt"],
    errors="coerce",
)

for field, efficiency in [
    ("p10_mt", 0.005),
    ("p50_mt", 0.020),
    ("p90_mt", 0.054),
]:
    observed = pd.to_numeric(
        internal_check[field],
        errors="coerce",
    )

    expected = theoretical * efficiency

    internal_check[f"{field}_expected"] = expected

    internal_check[f"{field}_matches_theoretical"] = (
        (observed - expected).abs() < 0.11
    )

display(
    internal_check[
        [
            "aquifer_name",
            "theoretical_storage_mt",
            "p10_mt",
            "p10_mt_expected",
            "p10_mt_matches_theoretical",
            "p50_mt",
            "p50_mt_expected",
            "p50_mt_matches_theoretical",
            "p90_mt",
            "p90_mt_expected",
            "p90_mt_matches_theoretical",
        ]
    ]
)

,aquifer_name,theoretical_storage_mt,p10_mt,p10_mt_expected,p10_mt_matches_theoretical,p50_mt,p50_mt_expected,p50_mt_matches_theoretical,p90_mt,p90_mt_expected,p90_mt_matches_theoretical
0,Boundary Lake-Osprey,0.000000,3.400000,0.000000,False,13.600000,0.000000,False,36.80000,0.000000,False
1,Dawson,0.000000,18.000000,0.000000,False,71.600000,0.000000,False,193.40000,0.000000,False
2,Monias-Beg,0.000000,9.200000,0.000000,False,36.700000,0.000000,False,99.00000,0.000000,False
3,Parkland-Stoddart,0.000000,15.600000,0.000000,False,62.300000,0.000000,False,168.20000,0.000000,False
4,Flatrock-Monias,2316.891605,11.500000,11.584458,True,46.300000,46.337832,True,125.10000,125.112147,True
5,Peejay-Weasel,1514.785508,7.600000,7.573928,True,30.300000,30.295710,True,81.80000,81.798417,True
6,Fireweed-Martin,1125.933804,5.600000,5.629669,True,22.500000,22.518676,True,60.80000,60.800425,True
7,Boundary Lake,2433.219081,12.166095,12.166095,True,48.664382,48.664382,True,131.39383,131.393830,True
8,Doe-Stoddart,0.000000,24.900000,0.000000,False,99.600000,0.000000,False,268.80000,0.000000,False
9,Ladyfern-Ring,507.318832,2.500000,2.536594,True,10.100000,10.146377,True,27.40000,27.395217,True


### Aquifer Theoretical Storage Field Consistency

The aquifer shapefiles contain 25 logical aquifers.

For 17 aquifers, `Th_SCap_MT` is nonzero. In every one of these cases,
the effective storage values are internally consistent with the reported
storage-efficiency factors:

- P10 = theoretical storage × 0.5%
- P50 = theoretical storage × 2%
- P90 = theoretical storage × 5.4%

Eight aquifers instead contain `Th_SCap_MT = 0.0` while retaining nonzero
P10, P50, and P90 effective storage estimates.

These zero values should therefore not be interpreted as zero theoretical
storage potential. They appear to represent missing or unpopulated theoretical
storage attributes in those shapefiles.

The affected aquifers are:

- Boundary Lake-Osprey
- Dawson
- Monias-Beg
- Parkland-Stoddart
- Doe-Stoddart
- Beg-Siphon
- Blueberry-Two Rivers
- Brassey-Cutbank

For later ingestion, the raw shapefile value should be preserved, but
`Th_SCap_MT = 0` should be flagged as a source-data anomaly where effective
storage values are nonzero.

In [74]:
# ---------------------------------------------------------------------------
# Recover implied theoretical storage for SHP zero-value anomalies
# ---------------------------------------------------------------------------

zero_theoretical_names = theoretical_anomalies["aquifer_name"]

zero_theoretical_check = aquifer_comparison[
    aquifer_comparison["aquifer_name"].isin(zero_theoretical_names)
].copy()

zero_theoretical_check["implied_theoretical_p10_mt"] = (
    pd.to_numeric(
        zero_theoretical_check[
            "P10 Effective Storage Potential at 0.5% (Mt)"
        ],
        errors="coerce",
    ) / 0.005
)

zero_theoretical_check["implied_theoretical_p50_mt"] = (
    pd.to_numeric(
        zero_theoretical_check[
            "P50 Effective Storage Potential at 2% (Mt)"
        ],
        errors="coerce",
    ) / 0.02
)

zero_theoretical_check["implied_theoretical_p90_mt"] = (
    pd.to_numeric(
        zero_theoretical_check[
            "P90 Effective Storage Potential at 5.4% (Mt)"
        ],
        errors="coerce",
    ) / 0.054
)

display(
    zero_theoretical_check[
        [
            "aquifer_name",
            "Aquifer Name",
            "theoretical_storage_mt",
            "implied_theoretical_p10_mt",
            "implied_theoretical_p50_mt",
            "implied_theoretical_p90_mt",
        ]
    ]
)

,aquifer_name,Aquifer Name,theoretical_storage_mt,implied_theoretical_p10_mt,implied_theoretical_p50_mt,implied_theoretical_p90_mt
0,Boundary Lake-Osprey,Boundary Lake-Osprey,0.0,681.053250,681.053250,681.053250
1,Dawson,Dawson,0.0,3581.872052,3581.872052,3581.872052
2,Monias-Beg,Monias-Beg,0.0,1833.983400,1833.983400,1833.983400
3,Parkland-Stoddart,Parkland-Stoddart,0.0,3115.072595,3115.072595,3115.072595
8,Doe-Stoddart,Doe-Stoddart,0.0,4978.185410,4978.185410,4978.185410
22,Beg-Siphon,Beg-Siphon,0.0,1186.848503,1186.848503,1186.848503
23,Blueberry-Two Rivers,Blueberry-Two Rivers,0.0,22030.144289,22030.144289,22030.144289
24,Brassey-Cutbank,Brassey-Cutbank,0.0,4785.616328,4785.616328,4785.616328


### Recovery of Missing Aquifer Theoretical Storage

Eight aquifer shapefiles contain `Th_SCap_MT = 0.0` despite having nonzero
effective-storage estimates.

For each of these aquifers, Appendix C P10, P50, and P90 values independently
back-calculate to the same theoretical-storage estimate using the atlas
efficiency factors:

- P10 / 0.005
- P50 / 0.020
- P90 / 0.054

This confirms that the zero `Th_SCap_MT` values in these shapefiles represent
missing or unpopulated source attributes rather than true zero storage
potential.

For later data integration:

- preserve the raw shapefile value for provenance;
- flag cases where theoretical storage is zero while effective storage is
  nonzero;
- recover or populate theoretical storage from Appendix C where available; and
- retain the Appendix C value as the documented source for the recovered
  estimate.

In [75]:
# ---------------------------------------------------------------------------
# Diagnose unmatched Blueberry-Buick aquifer name
# ---------------------------------------------------------------------------

name_mask = (
    aquifers["Aquifer Name"]
    .astype(str)
    .str.contains(
        "Blueberry|Buick",
        case=False,
        na=False,
    )
)

display(
    aquifers.loc[
        name_mask,
        [
            "Formation",
            "Aquifer Name",
            "Type",
            "Thickness Range (m)",
            "Pressure Range (MPa)",
            "Temperature Range (C)",
            "Porosity Range (%)",
            "CO2 Phase",
            "P10 Effective Storage Potential at 0.5% (Mt)",
            "P50 Effective Storage Potential at 2% (Mt)",
            "P90 Effective Storage Potential at 5.4% (Mt)",
        ],
    ]
)

,Formation,Aquifer Name,Type,Thickness Range (m),Pressure Range (MPa),Temperature Range (C),Porosity Range (%),CO2 Phase,P10 Effective Storage Potential at 0.5% (Mt),P50 Effective Storage Potential at 2% (Mt),P90 Effective Storage Potential at 5.4% (Mt)
7,Nikanassin-Dunlevy,Blueberry-Two Rivers,Aquifer,10-110,7.0-15.0,26-60,7-14%,Mostly Supercritical,110.150721,440.602886,1189.627792
20,Debolt,Blueberry-Buick,Aquifer,<2-30,15.0-25.0,42-99,4-17%,Mostly Supercritical,7.670672,30.682688,82.843257


In [76]:
# ---------------------------------------------------------------------------
# Normalize aquifer join keys and repeat SHP ↔ Appendix C join
# ---------------------------------------------------------------------------

aquifer_shp_summary["aquifer_name_key"] = (
    aquifer_shp_summary["aquifer_name"]
    .astype(str)
    .str.strip()
)

aquifers["aquifer_name_key"] = (
    aquifers["Aquifer Name"]
    .astype(str)
    .str.strip()
)

aquifer_comparison = aquifer_shp_summary.merge(
    aquifers,
    on="aquifer_name_key",
    how="left",
    validate="one_to_one",
    suffixes=("_shp", "_xlsx"),
)

print(f"Comparison rows: {len(aquifer_comparison)}")
print(
    "Missing Appendix C matches:",
    aquifer_comparison["Aquifer Name"].isna().sum(),
)

Comparison rows: 25
Missing Appendix C matches: 0


In [77]:
# ---------------------------------------------------------------------------
# Re-run aquifer storage consistency comparison
# ---------------------------------------------------------------------------

comparison_fields = {
    "p10_mt": "P10 Effective Storage Potential at 0.5% (Mt)",
    "p50_mt": "P50 Effective Storage Potential at 2% (Mt)",
    "p90_mt": "P90 Effective Storage Potential at 5.4% (Mt)",
}

results = []

for shp_field, xlsx_field in comparison_fields.items():
    left = pd.to_numeric(
        aquifer_comparison[shp_field],
        errors="coerce",
    )

    right = pd.to_numeric(
        aquifer_comparison[xlsx_field],
        errors="coerce",
    )

    match = (
        (left - right).abs() < 0.11
    ) | (
        left.isna() & right.isna()
    )

    results.append(
        {
            "shapefile_field": shp_field,
            "appendix_c_field": xlsx_field,
            "matches": int(match.sum()),
            "rows": len(match),
            "match_pct": 100 * match.mean(),
        }
    )

aquifer_value_summary = pd.DataFrame(results)

display(aquifer_value_summary)

,shapefile_field,appendix_c_field,matches,rows,match_pct
0,p10_mt,P10 Effective Storage Potential at 0.5% (Mt),22,25,88.0
1,p50_mt,P50 Effective Storage Potential at 2% (Mt),22,25,88.0
2,p90_mt,P90 Effective Storage Potential at 5.4% (Mt),22,25,88.0


### Aquifer SHP ↔ Appendix C Interpretation

All 25 aquifer shapefiles were successfully linked one-to-one with the 25
Appendix C aquifer records after trimming source-name whitespace.

#### Spatial structure

Most aquifers are represented by a single spatial feature, but two aquifer
files contain three polygon features.

These multi-feature records do not represent additive storage capacities.
Storage estimates may be repeated across multiple spatial components, while
CO2 phase can vary between components.

Logical aquifer identity should therefore be kept separate from individual
spatial-feature identity.

#### Schema variation

Seven source schemas occur across the 25 aquifer shapefiles, but most
differences are superficial:

- capitalization differences;
- field-name capitalization differences;
- field ordering differences; and
- inconsistent `Shape_*` naming.

Three Halfway aquifers additionally contain a descriptive `NAME` field.
One Middle Devonian aquifer contains a `REC_ID` field.

#### Storage attributes

The shapefile storage fields correspond to:

- `Th_SCap_MT` / `TH_SCap_MT`: theoretical storage potential;
- `Eff_SCap_h`: P10 effective storage potential at 0.5%;
- `Eff_SCap_2` / `Eff_Scap_2`: P50 effective storage potential at 2%;
- `Eff_SCap_5` / `Eff_Scap_5`: P90 effective storage potential at 5.4%; and
- `CO2_PHASE` / `CO2_Phase`: CO2 phase.

For 22 of 25 aquifers, the SHP P10, P50, and P90 values agree with Appendix C
within rounding tolerance.

The three remaining mismatches are:

- Beg-Siphon;
- Blueberry-Two Rivers; and
- Brassey-Cutbank.

For each of these three aquifers, the ratio between SHP and Appendix C storage
values is approximately constant across P10, P50, and P90, indicating that the
difference originates in the underlying aquifer storage estimate rather than
the efficiency factors.

#### Theoretical storage anomaly

Eight aquifer shapefiles contain `Th_SCap_MT = 0.0` despite having nonzero
effective-storage values.

For these aquifers, Appendix C P10, P50, and P90 values independently
back-calculate to a consistent nonzero theoretical-storage estimate.

The zero shapefile values should therefore be treated as unpopulated or
missing source attributes rather than true zero theoretical storage.

#### Source hierarchy

For aquifers, Appendix C appears to be the richer tabular source because it
contains reservoir-property ranges and consistently usable storage estimates.

The shapefiles provide the spatial representation and useful source-native
storage attributes, but contain several schema inconsistencies, geometry
warnings, spelling inconsistencies, and missing theoretical-storage values.

For later ingestion, raw SHP and Appendix C values should both remain
traceable, with reconciliation deferred to the standardized layer.

## Dataset Exploration Summary

The Northeast BC Geological Carbon Capture and Storage Atlas provides a substantially richer and more internally connected public storage dataset than the Alberta sequestration-agreement layer explored previously.

For depleted and near-depleted pools, the master pool shapefile appears to be the primary native geospatial source. Appendix C largely reproduces the same pool attributes in a curated tabular form while adding useful screening classifications such as current storage candidate, future storage candidate, and oil-pool CO2-EOR evaluation.

For saline aquifers, the source structure is different. The shapefiles provide the spatial representation, but Appendix C is the richer tabular source because it contains reservoir-property ranges and more consistently usable storage estimates. The aquifer shapefiles also contain several source-quality issues, including inconsistent schemas, CRS variation, spelling differences, geometry warnings, repeated spatial features, and some unpopulated theoretical-storage fields.

A consistent source relationship can nevertheless be reconstructed:

- logical storage objects should be separated from individual spatial features;
- source-native identifiers and names should be preserved for traceability;
- pool and aquifer records require different ingestion logic;
- Appendix C and the shapefiles should remain linked rather than treated as independent datasets; and
- raw values should be preserved before any normalization or reconciliation.

Overall, the Northeast BC atlas appears suitable for integration into a unified geological-storage GeoPackage, provided that provenance, logical-versus-spatial identity, schema inconsistencies, and source-specific anomalies are explicitly represented.

The next exploration will examine the Atlantic Canadian storage dataset. Together with the Alberta and Northeast BC sources, this should provide enough contrast to determine what common storage ontology can be supported across publicly available Canadian datasets and which sources can be safely merged into a standardized national GeoPackage.

The Alberta sequestration-agreement dataset will likely require different treatment because it describes tenure and agreement geography but does not currently provide comparable storage-potential or capacity estimates.